<div align="center">

**Türkçe:** Bu eğitim Türkçe yürütülmektedir. [Bu Jupyter Notebook'un Türkçesini Colab'da açın](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-05/NB05_zaman_serisi_ve_ekg_yapay_zekasi.ipynb) · [Türkçe ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-05/ders.html)

# Week 05: Time series analysis and ECG artificial intelligence

**Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists**<br>
Turkish Society of Cardiology · Prof. Dr. Utku Kose

[Week page](README.md) · [Lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html) · [Interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lab.html) · [PDF notes](Week05_Lecture_Notes.pdf) · [Open in Colab](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-05/NB05_time_series_and_ecg_ai.ipynb)

</div>

**This notebook has two parts.** In the first, an artificial intelligence (AI) assistant writes the code from a prompt, a written instruction. In the second, you write the code yourself in Python. Both parts reach the same results with the same data, so choose one of them at each practice stop of the lecture page.

**Together with the lecture page.** You come to this Jupyter notebook from the practice stops of the lecture page. At the end of every stop, a link takes you back to the lecture page, to the place where reading continues.

| | For whom? | What you do |
|---|---|---|
| **[Part 1: Prompt path](#part-1)** | Participants who do not write code | An AI assistant writes the code from ready prompts; you run it and check the result. |
| **[Part 2: Python path](#part-2)** | Participants who want to learn Python | You write and run the code step by step, with exercises. |

**Where it runs.** The same notebook runs in Google Colab, with nothing to install, and in Jupyter on your own computer (Anaconda; see SETUP.md in the course folder). Choose the one you prefer. In Colab, save your own copy first with *File → Save a copy in Drive*; otherwise the changes are lost when the tab is closed. Training the convolutional network takes about a minute and the recurrent network about half a minute on an ordinary laptop; no graphics card is needed. Run Section 0 first, whichever part you choose. It prepares the course files and the packages. A link of the lecture page opens the notebook at a stop or a section: There, run the first cell, the preparation cell or the catch-up cell. It also runs Section 0 if this has not happened in the session.

**Data and privacy.** Every data set here is synthetic or comes from a public research database. Real patient data belongs only on your own computer, under the rules of your institution and with the approval of an ethics committee, and never in a cloud notebook or in an AI chat.

## 0. Setup

The first cell finds the course folder. If the notebook was opened on its own, for example in Google Colab, it downloads the course package `vclinic` and the data files from GitHub. The second cell loads the packages, chooses the language of the messages and checks the optional packages of the week. The second cell downloads the 12-lead ECGs of the course from PhysioNet: the recordings at 100 Hz of folds 1, 2, 9 and 10 of PTB-XL, 8,737 recordings with their two tables, about 216 MB. The first run takes a few minutes; later runs skip the files that are already there. PTB-XL is published under the Creative Commons Attribution 4.0 International License (CC BY 4.0). Every use cites Wagner et al., Scientific Data, 2020, and the data set on PhysioNet.

In [ ]:
#@title Course files: run this cell (there is no need to open it)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # A downloaded copy, for example in Colab: the files of the package are refreshed.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Course files downloaded to", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Setup: packages, language and plotting style
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import copy
import math
from collections import Counter

from sklearn.metrics import roc_auc_score, roc_curve
from vclinic import ptbxl, beats, holter

try:
    import torch
    import torch.nn as nn
except ImportError as error:
    raise ImportError("PyTorch is not installed. Install it as SETUP.md describes, then restart the kernel.") from error
import vclinic
from vclinic import teach

if tuple(int(x) for x in vclinic.__version__.split(".")) < (1, 6, 0):
    raise ImportError("The course package vclinic is older than this notebook. Update the course folder (git pull, or download the ZIP file again) and restart the kernel.")
teach.set_language("en")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026
torch.manual_seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))
print("PyTorch", torch.__version__)
ptbxl.download()
print("vclinic version", vclinic.__version__)
print("Course folder:", ROOT)
for name in ["ipywidgets"]:
    try:
        importlib.import_module(name)
        print(f"{name}: installed")
    except ImportError:
        print(f"{name}: not installed (optional; the notebook uses a built-in fallback)")

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
teach.register_answers({
    "Exercise 5.1": (4, 0),
    "Exercise 5.2": (4, 0),
    "Exercise 5.3": (1.168, 0.001),
    "Exercise A": (60, 0),
    "Exercise B": (0.704, 0.001),
    "Exercise C": (60396, 0),
    "Exercise D": (0.703, 0.03),
    "Exercise E": (3558, 0),
    "Exercise F": (1.168, 0.001),
    "Exercise G": (215, 0),
    "Exercise H": (217, 0),
    "Exercise I": (14.7, 0.05),
    "Exercise J": (0.882, 0.001),
    "Step 1.1": (0.914, 0.001),
    "Step 1.2": (0.881, 0.03),
    "Step 1.3": (0.975, 0.02),
    "Step 1.4": (0.636, 0.06),
    "Step 1.5": (0.848, 0.03),
    "Step 2.1": (19, 0),
    "Step 2.2": (9, 0),
    "Step 2.3": (51, 0),
    "Step 3.1": (519, 0),
    "Step 3.2": (18.0, 0.05),
    "Step 3.3": (75, 0),
})
print("Expected answers are ready.")

In [ ]:
#@title Tables of the exercise prompts (run this cell; the Python path does not need to read it)
def exercise_table(key):
    """The table "data" of the prompt of an exercise, built from the data of the Python path."""
    g = globals()
    k = key
    leads = list(beats.LEADS)
    if k == "5.1":
        return pd.concat([pd.DataFrame({"time": np.round(np.arange(1000) / 100, 2)}), pd.DataFrame(g["x"].T, columns=leads)], axis=1)
    if k == "5.2":
        return g["labels_vc"].reset_index().rename(columns={"patient_id": "patient"})
    if k == "5.3":
        return pd.DataFrame({"patient": g["labels_vc"].index, "lead II": list(g["X"][:, 1])})
    if k == "A":
        return pd.concat([pd.DataFrame({"record": g["ecgs"]["ecg_id"].to_numpy()[g["te"]]}),
                          g["labels"].iloc[np.where(g["te"])[0]].reset_index(drop=True)], axis=1)
    if k in ("B", "D", "F", "G", "H", "I", "J"):
        return g["_exercise_tables"][k]()
    raise KeyError(k)


def _st_rows(mask):
    ix = np.where(mask)[0]
    st = st_table.set_index("ecg_id").loc[ecgs["ecg_id"].to_numpy()[ix]]
    return pd.concat([pd.DataFrame({"record": ecgs["ecg_id"].to_numpy()[ix],
                                    "sex": np.where(ecgs["sex"].to_numpy()[ix] == "M", "M", "F"),
                                    "age": ecgs["age"].to_numpy()[ix].astype(int),
                                    "acute infarction": ecgs["acute_mi"].to_numpy()[ix].astype(int)}),
                      st[leads_order].reset_index(drop=True).round(3)], axis=1)


leads_order = list(beats.LEADS)
_exercise_tables = {
    "B": lambda: pd.DataFrame({"record": ecgs["ecg_id"].to_numpy()[te], "RR": [np.round(rr_list[k], 2) for k in np.where(te)[0]],
                               "AF": labels["AFIB"].to_numpy()[te]}),
    "D": lambda: pd.concat([pd.DataFrame({"record": ecgs["ecg_id"].to_numpy()[te]}), labels.iloc[np.where(te)[0]].reset_index(drop=True),
                            pd.DataFrame(np.round(test_prob, 4), columns=[f"probability {c}" for c in RHYTHMS])], axis=1),
    "F": lambda: pd.concat([pd.DataFrame({"time": np.arange(-30, 56) * 10}), pd.DataFrame(np.round(case_beat.T, 3), columns=leads_order)], axis=1),
    "G": lambda: _st_rows(va | te),
    "H": lambda: pd.DataFrame({"record": qt_table["ecg_id"], "QT (ms)": qt_table["qt_ms"].round(2), "RR (s)": qt_table["rr_s"].round(4)}),
    "I": lambda: pd.DataFrame({"time": np.round(holter_beats["time"], 3), "RR": np.round(holter_beats["RR"], 3)}),
    "J": lambda: pd.DataFrame({"record": ecgs["ecg_id"].to_numpy(), "patient": ecgs["patient_id"].to_numpy().astype(int),
                               "set": [{"training": "training", "validation": "validation", "test": "test"}[s] for s in split],
                               "RR": [np.round(r, 2) for r in rr_list], "AF": labels["AFIB"].to_numpy()}),
}
print("The tables of the exercise prompts are ready.")

<a id="part-1"></a>

# Part 1: Prompt path

**For whom?** This part is for participants who do not write code. Here you type no code. An artificial intelligence (AI) assistant writes it from a ready prompt, a written instruction; you run the code and check the result.

**What you need.** An AI assistant that can write Python code is enough. Gemini is built into Google Colab: Open it with the Gemini button at the top of the window. ChatGPT, Claude, Copilot or Gemini in another browser tab work as well.

**How a step works.**

1. Read the goal of the step.
2. Run the data cell under the prompt. It prepares the table "data" that the prompt describes and shows its first and last rows.
3. Copy the prompt: Select it with the mouse and press Ctrl+C (Cmd+C on a Mac).
4. Paste it into the AI assistant and send it.
5. Copy the code that the assistant returns, paste it into the cell under the data cell in place of the line that starts with `#`, and run the cell with the ▶ button or with Shift+Enter.
6. Run the check cell below it. It prints *correct ✓* or *not yet ✗*.
7. If it prints *not yet*, send the repair prompt below to the assistant together with the code. If the result is still wrong, run the *Ready code* cell. It always gives the correct result, so the next step can start.

**"data" and "answer".** As in Week 1, the data cell of each step prepares the table "data" that the prompt describes. The assistant keeps the result under the name "answer", and the check cell reads it from there.

**At the start of every stop** run its preparation cell once. It loads the data of the stop and also runs Section 0 if this has not happened in the session, so a stop can also be started on another day or straight from the lecture page.

**Repair prompt** (for code that runs but does not pass the check):

> The code below was written for this task: [paste the Task line of the prompt]. It runs, but the check of the course prints "not yet". Find the error, explain it in two sentences and return the corrected code. The result must still be kept under the name "answer".  
>
> [paste the code here]  

**Prompt for an error message** (for code that stops with an error):

> Running the code below in a Jupyter notebook gives the error message that follows. Explain the cause in plain language and return the corrected code.  
>
> [paste the code here]  
>
> [paste the error message here]  

All prompts of the week are also in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-05/PROMPTS.md), where GitHub shows a copy button next to each one.

**A rule that never changes.** The prompts contain only the synthetic or public data of the course. Never paste real patient data into an AI assistant.

## Stop 1: Mini project: Rhythm, from a rule to the nine classes of the PhysioNet 2020 Challenge

*Lecture: [Practice stop 1](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#stop-1)* · *The same result on the Python path: Part 2, Sections 1 to 5*

This stop is the mini project of the session. The 12-lead ECGs of PTB-XL are read first as a cardiologist reads the rhythm, from the RR intervals, and then by a one-dimensional convolutional network that learns six labels at once. The network is trained on folds 1 and 2, its thresholds are chosen on fold 9, and both the rule and the network are judged on fold 10. The last step judges the same network trained for the nine classes of the PhysioNet 2020 Challenge.

**At the end of this stop:** The sensitivity of the rule for AF (0.914), the mean AUC of the network on fold 9 (about 0.881), the AUC of the network for AF on fold 10 (about 0.975) the positive predictive value of its AF alarm with the threshold chosen on fold 9 (about 0.636) and the mean AUC of the nine classes of the Challenge on fold 10 (about 0.848).

In [ ]:
#@title Preparation of Stop 1: run this cell once (it loads the data of the stop; if Section 0 has not run in this session, it runs it first)
def _catch_up(upto, done, name="NB05_time_series_and_ecg_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
if "ecgs" not in globals():
    ecgs = ptbxl.table()
    signals = ptbxl.signals(ecgs)
    RHYTHMS = ["SR", "AFIB", "STACH", "SBRAD", "SARRH", "PVC"]
    labels = ptbxl.labels(ecgs, RHYTHMS)
    split = ecgs["split"].to_numpy()
    tr, va, te = split == "training", split == "validation", split == "test"

    r_peak_list = []
    for start in range(0, len(signals), 500):
        r_peak_list += beats.r_peaks_many(signals[start:start + 500])
    rr_list = [np.diff(p) / 100 for p in r_peak_list]
    cv = np.array([rr.std(ddof=1) / rr.mean() if len(rr) >= 3 else np.nan for rr in rr_list])


def macro_auc(y_true, prob):
    """Mean of the AUCs of the labels: one AUC per column."""
    return float(np.mean([roc_auc_score(y_true[:, k], prob[:, k]) for k in range(y_true.shape[1])]))


def youden_threshold(y_true, prob):
    """The threshold at which sensitivity + specificity is highest."""
    fpr, tpr, thresholds = roc_curve(y_true, prob)
    return float(thresholds[np.argmax(tpr - fpr)])


def se_sp_ppv(y_true, alarm):
    """Sensitivity, specificity and positive predictive value of a yes-or-no decision."""
    y_true, alarm = np.asarray(y_true, dtype=bool), np.asarray(alarm, dtype=bool)
    tp, fp = int((alarm & y_true).sum()), int((alarm & ~y_true).sum())
    return tp / y_true.sum(), 1 - fp / (~y_true).sum(), tp / max(1, tp + fp)


def conv_block(c_in, c_out, k):
    """One layer: a convolution along time with stride 2, batch normalisation and ReLU."""
    return [nn.Conv1d(c_in, c_out, k, stride=2, padding=k // 2), nn.BatchNorm1d(c_out), nn.ReLU()]


class RhythmNet(nn.Module):
    """Four convolutions over the 12 leads, the mean of every channel over time and one output per label."""

    def __init__(self, n_labels=6):
        super().__init__()
        self.features = nn.Sequential(*conv_block(12, 32, 7), *conv_block(32, 64, 5),
                                      *conv_block(64, 64, 5), *conv_block(64, 128, 3))
        self.out = nn.Linear(128, n_labels)

    def forward(self, x):
        return self.out(self.features(x).mean(dim=2))


def predict(net, *inputs, batch=256):
    """The probabilities of the labels, computed in batches without training."""
    net.eval()
    with torch.no_grad():
        return torch.cat([torch.sigmoid(net(*[x[i:i + batch] for x in inputs]))
                          for i in range(0, len(inputs[0]), batch)]).numpy()


def _course_rhythmnet():
    """The network of the course, trained once (about a minute): the probabilities of the six labels on folds 9 and 10."""
    if "_rhythm_cache" in globals():
        return _rhythm_cache
    global x_all, y_all, x_train, y_train, x_val, x_test, y_val, y_test
    x_all = torch.tensor(np.clip(signals, -5, 5))
    y_all = torch.tensor(np.array(labels, dtype=np.float32))
    x_train, y_train = x_all[tr], y_all[tr]
    x_val, x_test = x_all[va], x_all[te]
    y_val, y_test = labels.to_numpy()[va], labels.to_numpy()[te]

    torch.manual_seed(RANDOM_STATE)
    rhythm_net = RhythmNet(len(RHYTHMS))
    EPOCHS, BATCH = 15, 64
    optimizer = torch.optim.Adam(rhythm_net.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.003, total_steps=EPOCHS * math.ceil(len(x_train) / BATCH))
    loss_fn = nn.BCEWithLogitsLoss()
    best_auc, best_epoch, val_history = -1.0, 0, []
    for epoch in range(1, EPOCHS + 1):
        rhythm_net.train()
        order = torch.randperm(len(x_train))
        for i in range(0, len(order), BATCH):
            b = order[i:i + BATCH]
            optimizer.zero_grad()
            loss = loss_fn(rhythm_net(x_train[b]), y_train[b])
            loss.backward()
            optimizer.step()
            scheduler.step()
        val_auc = macro_auc(y_val, predict(rhythm_net, x_val))
        val_history.append(val_auc)
        if val_auc > best_auc:
            best_auc, best_epoch, best_state = val_auc, epoch, copy.deepcopy(rhythm_net.state_dict())
        print(f"epoch {epoch:2d} · mean AUC on fold 9: {val_auc:.3f}")
    rhythm_net.load_state_dict(best_state)
    val_prob, test_prob = predict(rhythm_net, x_val), predict(rhythm_net, x_test)
    globals()["_rhythm_cache"] = (val_prob, test_prob)
    return _rhythm_cache


def _course_net9():
    """The network of the course for the nine classes of CPSC 2018, trained once (about a minute): the labels and the probabilities on fold 10."""
    if "_nine_cache" in globals():
        return _nine_cache
    x_all = torch.tensor(np.clip(signals, -5, 5))
    y_all = torch.tensor(np.array(labels, dtype=np.float32))
    x_train, y_train = x_all[tr], y_all[tr]
    x_val, x_test = x_all[va], x_all[te]
    y_val, y_test = labels.to_numpy()[va], labels.to_numpy()[te]

    CPSC = {"NORM": "Normal", "AFIB": "AF", "1AVB": "I-AVB", "CLBBB": "LBBB", "CRBBB": "RBBB",
            "PAC": "PAC", "PVC": "PVC", "STD_": "STD", "STE_": "STE"}
    labels9 = ptbxl.labels(ecgs, list(CPSC))
    y9_all = torch.tensor(np.array(labels9, dtype=np.float32))
    y9_val, y9_test = labels9.to_numpy()[va], labels9.to_numpy()[te]


    def fit_labels(net, y_train, y_val, epochs=15, batch=64):
        """Trains a multi-label network on folds 1 and 2 and keeps the epoch with the best mean AUC on fold 9."""
        optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.003, total_steps=epochs * math.ceil(len(x_train) / batch))
        loss_fn = nn.BCEWithLogitsLoss()
        best_auc, best_epoch, best_state = -1.0, 0, None
        for epoch in range(1, epochs + 1):
            net.train()
            order = torch.randperm(len(x_train))
            for i in range(0, len(order), batch):
                b = order[i:i + batch]
                optimizer.zero_grad()
                loss = loss_fn(net(x_train[b]), y_train[b])
                loss.backward()
                optimizer.step()
                scheduler.step()
            val_auc = macro_auc(y_val, predict(net, x_val))
            if val_auc > best_auc:
                best_auc, best_epoch, best_state = val_auc, epoch, copy.deepcopy(net.state_dict())
        net.load_state_dict(best_state)
        return best_epoch, best_auc

    torch.manual_seed(RANDOM_STATE)
    net9 = RhythmNet(len(CPSC))
    best_epoch9, best_auc9 = fit_labels(net9, y9_all[tr], y9_val)
    test_prob9 = predict(net9, x_test)
    table9 = pd.DataFrame({"fold 1-2": labels9.to_numpy()[tr].sum(axis=0),
                           "fold 10": y9_test.sum(axis=0).astype(int),
                           "AUC": [round(roc_auc_score(y9_test[:, k], test_prob9[:, k]), 3) for k in range(len(CPSC))]},
                          index=list(CPSC.values()))
    globals()["_nine_cache"] = (labels9.rename(columns=CPSC), test_prob9)
    return _nine_cache

_SETS = {"training": "training", "validation": "validation", "test": "test"}
print(len(ecgs), "recordings;", {s: int((split == s).sum()) for s in ("training", "validation", "test")})
print("Stop 1 is ready.")

### Step 1.1: A rule from the RR intervals

**Goal.** Test the first half of the clinical rule for AF, irregularly irregular RR intervals, on the test fold.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know how well a simple rule from the RR intervals detects atrial fibrillation (AF) in 12-lead ECGs of 10 seconds.  
> Data: The notebook holds a table named "data" with the 2,198 recordings of the test fold of PTB-XL, a public collection of clinical 12-lead ECGs of 10 seconds. Each row is a recording: "record" gives its number, "RR" holds its RR intervals in seconds, found from the R peaks of all 12 leads, and "AF" is 1 when the cardiologists described atrial fibrillation and 0 otherwise.  
> Task: For every recording with at least three RR intervals, compute the coefficient of variation: the standard deviation of the intervals, with the number of intervals minus one in the denominator, divided by their mean. Call a recording AF when its coefficient of variation is above 0.10. Compare the rule with the column "AF" and compute its sensitivity, its specificity and its positive predictive value. Also draw the coefficients of variation of the recordings with and without AF as two histograms, with a vertical line at 0.10.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of alarms and the three values, and keep the sensitivity, with three decimals, under the name "answer".  
> Test: The alarms must equal the true alarms plus the false alarms, and the sensitivity must equal the true alarms divided by the recordings with AF.  

In [ ]:
#@title Data of step 1.1: run this cell (it prepares the table "data" and shows it)
_i = np.where(te)[0]
data = pd.DataFrame({"record": ecgs["ecg_id"].to_numpy()[_i], "RR": [np.round(rr_list[k], 2) for k in _i],
                     "AF": labels["AFIB"].to_numpy()[_i]})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.1
teach.check_step("Step 1.1", "answer", globals())

In [ ]:
#@title Ready code of step 1.1 (run it only if the code of the assistant does not pass the check)
_i = np.where(te)[0]
data = pd.DataFrame({"record": ecgs["ecg_id"].to_numpy()[_i], "RR": [np.round(rr_list[k], 2) for k in _i],
                     "AF": labels["AFIB"].to_numpy()[_i]})
cv = np.array([np.std(r, ddof=1) / np.mean(r) if len(r) >= 3 else np.nan for r in data["RR"]])
alarm = cv > 0.10
af = data["AF"].to_numpy() == 1
tp, fp = int((alarm & af).sum()), int((alarm & ~af).sum())
answer = round(tp / af.sum(), 3)
specificity = 1 - fp / (~af).sum()
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(cv[~af], bins=np.linspace(0, 0.5, 51), color="#1F5F8B", alpha=0.6, label="no AF")
ax.hist(cv[af], bins=np.linspace(0, 0.5, 51), color="#9B1B1B", alpha=0.7, label="AF")
ax.axvline(0.10, color="k", ls="--", lw=1)
ax.set_xlabel("coefficient of variation of the RR intervals")
ax.set_ylabel("recordings")
ax.legend()
plt.show()
print("Alarms:", int(alarm.sum()), "· sensitivity", answer, "· specificity", round(specificity, 3),
      "· PPV", round(tp / max(1, tp + fp), 3))
teach.check_step("Step 1.1", "answer", globals())

**What the result shows.** The rule finds 91 percent of the recordings with AF, but more than half of its alarms are false: Premature beats and sinus arrhythmia make the RR intervals irregular as well. The histograms show the overlap that no threshold can remove.

### Step 1.2: A one-dimensional network for six labels

**Goal.** Train a network that reads the whole signal of the 12 leads and gives six labels at once. The training takes about a minute.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to train a network that reads 12-lead ECGs of 10 seconds and recognises six findings of rhythm at once, and to know how well it separates them on new recordings.  
> Data: The notebook holds a table named "data" with 6,539 recordings of PTB-XL, a public collection of clinical 12-lead ECGs of 10 seconds. Each row is a recording: "set" gives its set ("training", folds 1 and 2, 4,356 recordings; "validation", fold 9, 2,183 recordings), and each cell of the column "ECG" holds the recording as an array of 12 leads × 1,000 samples in mV, recorded at 100 Hz, with the leads in the order I, II, III, aVR, aVL, aVF and V1 to V6. The six columns "SR", "AFIB", "STACH", "SBRAD", "SARRH" and "PVC" are 1 when the cardiologists gave the statement and 0 otherwise: sinus rhythm, atrial fibrillation, sinus tachycardia, sinus bradycardia, sinus arrhythmia and ventricular premature complexes.  
> Task: Build a one-dimensional convolutional network that gives one score to each of the six labels. It has four convolutional layers along time, each with stride 2 and each followed by batch normalisation and ReLU: 32 kernels of 7 samples over the 12 leads, then 64 kernels of 5, 64 kernels of 5 and 128 kernels of 3 samples. The mean of each of the 128 channels over time goes to a last linear layer with six outputs. Before training, limit the values of the signals to the range from −5 to 5 mV. Train it on the training recordings for 15 epochs, in shuffled groups of 64 recordings, with the binary cross-entropy of the six labels as the loss and the Adam optimiser. Use a one-cycle learning rate with a highest value of 0.003. After every epoch, compute the area under the ROC curve (AUC) of each label on the validation recordings and their mean, and keep the weights of the epoch where the mean is highest. Make the random choices the same in every run by using the number 2026 as the starting value.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of parameters, the mean AUC of every epoch and the best epoch, and keep the best mean AUC, with three decimals, under the name "answer".  
> Test: The network must have 59,622 parameters, and the mean AUC must lie between 0.5 and 1.  

In [ ]:
#@title Data of step 1.2: run this cell (it prepares the table "data" and shows it)
_i = np.where(tr | va)[0]
data = pd.concat([pd.DataFrame({"set": [_SETS[s] for s in split[_i]], "ECG": list(signals[_i])}),
                  labels.iloc[_i].reset_index(drop=True)], axis=1)
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.2
teach.check_step("Step 1.2", "answer", globals())

In [ ]:
#@title Ready code of step 1.2 (run it only if the code of the assistant does not pass the check)
_i = np.where(tr | va)[0]
data = pd.concat([pd.DataFrame({"set": [_SETS[s] for s in split[_i]], "ECG": list(signals[_i])}),
                  labels.iloc[_i].reset_index(drop=True)], axis=1)


def conv_block(c_in, c_out, k):
    """One layer: a convolution along time with stride 2, batch normalisation and ReLU."""
    return [nn.Conv1d(c_in, c_out, k, stride=2, padding=k // 2), nn.BatchNorm1d(c_out), nn.ReLU()]


class RhythmNet(nn.Module):
    """Four convolutions over the 12 leads, the mean of every channel over time and one output per label."""

    def __init__(self, n_labels=6):
        super().__init__()
        self.features = nn.Sequential(*conv_block(12, 32, 7), *conv_block(32, 64, 5),
                                      *conv_block(64, 64, 5), *conv_block(64, 128, 3))
        self.out = nn.Linear(128, n_labels)

    def forward(self, x):
        return self.out(self.features(x).mean(dim=2))


def predict(net, *inputs, batch=256):
    """The probabilities of the labels, computed in batches without training."""
    net.eval()
    with torch.no_grad():
        return torch.cat([torch.sigmoid(net(*[x[i:i + batch] for x in inputs]))
                          for i in range(0, len(inputs[0]), batch)]).numpy()

training = np.array(data["set"] == "training")
x = torch.tensor(np.clip(np.stack(data["ECG"]), -5, 5).astype(np.float32))
y = torch.tensor(np.array(data[RHYTHMS], dtype=np.float32))
x_train, y_train = x[training], y[training]
x_val, y_val = x[~training], y[~training].numpy()
torch.manual_seed(2026)
net = RhythmNet(6)
print("Parameters:", sum(p.numel() for p in net.parameters()))
optimiser = torch.optim.Adam(net.parameters(), lr=0.001)
schedule = torch.optim.lr_scheduler.OneCycleLR(optimiser, max_lr=0.003, total_steps=15 * math.ceil(len(x_train) / 64))
loss_fn = nn.BCEWithLogitsLoss()
best, best_epoch = -1.0, 0
for epoch in range(1, 16):
    net.train()
    order = torch.randperm(len(x_train))
    for start in range(0, len(order), 64):
        b = order[start:start + 64]
        optimiser.zero_grad()
        loss_fn(net(x_train[b]), y_train[b]).backward()
        optimiser.step()
        schedule.step()
    prob = predict(net, x_val)
    mean_auc = float(np.mean([roc_auc_score(y_val[:, k], prob[:, k]) for k in range(6)]))
    if mean_auc > best:
        best, best_epoch = mean_auc, epoch
    print("epoch", epoch, "mean AUC:", round(mean_auc, 3))
answer = round(best, 3)
print("Best epoch:", best_epoch, "· mean AUC:", answer)
teach.check_step("Step 1.2", "answer", globals())

**What the result shows.** In the run of the course the mean AUC on fold 9 rose from about 0.65 after the first epoch to about 0.881. The number differs a little from one computer to another, because the order of the groups and the arithmetic of the processors differ.

### Step 1.3: Each label on the test fold

**Goal.** Judge the network of the course label by label on recordings that it never saw.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know, for each of six findings of rhythm, how well a network separates the 12-lead ECGs with the finding from those without it.  
> Data: The notebook holds a table named "data" with the 2,198 recordings of the test fold of PTB-XL. Each row is a recording: "record" gives its number. The six columns "SR", "AFIB", "STACH", "SBRAD", "SARRH" and "PVC" are 1 when the cardiologists gave the statement and 0 otherwise: sinus rhythm, atrial fibrillation, sinus tachycardia, sinus bradycardia, sinus arrhythmia and ventricular premature complexes. The six columns "probability SR" to "probability PVC" hold the probabilities that a one-dimensional convolutional network, trained on folds 1 and 2, gave to the six labels.  
> Task: For each of the six labels, compute the area under the ROC curve (AUC) of its probability against its column of 0 and 1. Draw the six ROC curves in one figure, with the AUC of each in the legend.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print a small table with the label, the number of recordings with the label and the AUC, and keep the AUC of "AFIB", with three decimals, under the name "answer".  
> Test: Every AUC must lie between 0 and 1, and the numbers of recordings must add up to at least 2,198, because a recording can carry several labels.  

In [ ]:
#@title Data of step 1.3: run this cell (it prepares the table "data" and shows it)
_val_prob, _test_prob = _course_rhythmnet()
_i = np.where(te)[0]
data = pd.concat([pd.DataFrame({"record": ecgs["ecg_id"].to_numpy()[_i]}), labels.iloc[_i].reset_index(drop=True),
                  pd.DataFrame(np.round(_test_prob, 4), columns=[f"probability {c}" for c in RHYTHMS])], axis=1)
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.3
teach.check_step("Step 1.3", "answer", globals())

In [ ]:
#@title Ready code of step 1.3 (run it only if the code of the assistant does not pass the check)
_val_prob, _test_prob = _course_rhythmnet()
_i = np.where(te)[0]
data = pd.concat([pd.DataFrame({"record": ecgs["ecg_id"].to_numpy()[_i]}), labels.iloc[_i].reset_index(drop=True),
                  pd.DataFrame(np.round(_test_prob, 4), columns=[f"probability {c}" for c in RHYTHMS])], axis=1)
rows = []
fig, ax = plt.subplots(figsize=(4.6, 4.2))
for label in RHYTHMS:
    y, p = data[label], data[f"probability {label}"]
    auc = roc_auc_score(y, p)
    fpr, tpr, _ = roc_curve(y, p)
    ax.plot(fpr, tpr, lw=1.2, label=f"{label} {auc:.3f}")
    rows.append({"label": label, "recordings": int(y.sum()), "AUC": round(auc, 3)})
ax.plot([0, 1], [0, 1], color="#999999", lw=0.8, ls="--")
ax.set_xlabel("1 − specificity")
ax.set_ylabel("sensitivity")
ax.legend(fontsize=8)
plt.show()
print(pd.DataFrame(rows))
answer = round(roc_auc_score(data["AFIB"], data["probability AFIB"]), 3)
teach.check_step("Step 1.3", "answer", globals())

**What the result shows.** The network separates AF with an AUC of about 0.975, better than the rule of step 1.1, which reached 0.939 on the same recordings. Sinus arrhythmia is the weakest label: It depends on a slow change of the rate over several beats, which the network sees only through its average over the 10 seconds.

### Step 1.4: Thresholds chosen on fold 9

**Goal.** Turn the probabilities into decisions with one threshold per label, chosen on fold 9 and judged on fold 10.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to turn the probabilities of a network for six findings of rhythm into decisions, with one threshold for each finding chosen on recordings that are kept apart from the test.  
> Data: The notebook holds a table named "data" with the 4,381 recordings of folds 9 and 10 of PTB-XL. Each row is a recording: "record" gives its number and "set" its set ("validation", fold 9; "test", fold 10). The six columns "SR", "AFIB", "STACH", "SBRAD", "SARRH" and "PVC" are 1 when the cardiologists gave the statement and 0 otherwise: sinus rhythm, atrial fibrillation, sinus tachycardia, sinus bradycardia, sinus arrhythmia and ventricular premature complexes. The six columns "probability SR" to "probability PVC" hold the probabilities that a one-dimensional convolutional network, trained on folds 1 and 2, gave to the six labels.  
> Task: For each label, choose on the validation recordings the threshold of the probability at which sensitivity plus specificity is highest. Apply each threshold to the test recordings: A label is given when its probability reaches its threshold. Compute the sensitivity, the specificity and the positive predictive value of each label on the test recordings.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print a table with the label, its threshold and the three values, and keep the positive predictive value of "AFIB" on the test recordings, with three decimals, under the name "answer".  
> Test: The thresholds must come from the validation recordings only, and every sensitivity and specificity must lie between 0 and 1.  

In [ ]:
#@title Data of step 1.4: run this cell (it prepares the table "data" and shows it)
_val_prob, _test_prob = _course_rhythmnet()
_i = np.where(va | te)[0]
_prob = np.zeros((len(ecgs), len(RHYTHMS)))
_prob[va], _prob[te] = _val_prob, _test_prob
data = pd.concat([pd.DataFrame({"record": ecgs["ecg_id"].to_numpy()[_i], "set": [_SETS[s] for s in split[_i]]}),
                  labels.iloc[_i].reset_index(drop=True),
                  pd.DataFrame(np.round(_prob[_i], 4), columns=[f"probability {c}" for c in RHYTHMS])], axis=1)
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.4
teach.check_step("Step 1.4", "answer", globals())

In [ ]:
#@title Ready code of step 1.4 (run it only if the code of the assistant does not pass the check)
_val_prob, _test_prob = _course_rhythmnet()
_i = np.where(va | te)[0]
_prob = np.zeros((len(ecgs), len(RHYTHMS)))
_prob[va], _prob[te] = _val_prob, _test_prob
data = pd.concat([pd.DataFrame({"record": ecgs["ecg_id"].to_numpy()[_i], "set": [_SETS[s] for s in split[_i]]}),
                  labels.iloc[_i].reset_index(drop=True),
                  pd.DataFrame(np.round(_prob[_i], 4), columns=[f"probability {c}" for c in RHYTHMS])], axis=1)
val = data[data["set"] == "validation"]
test = data[data["set"] == "test"]
rows = []
for label in RHYTHMS:
    column = f"probability {label}"
    fpr, tpr, thresholds = roc_curve(val[label], val[column])
    threshold = float(thresholds[np.argmax(tpr - fpr)])
    y = test[label].to_numpy() == 1
    given = test[column].to_numpy() >= threshold
    tp, fp = int((given & y).sum()), int((given & ~y).sum())
    rows.append({"label": label, "threshold": round(threshold, 3),
                     "sensitivity": round(tp / y.sum(), 3), "specificity": round(1 - fp / (~y).sum(), 3),
                     "PPV": round(tp / max(1, tp + fp), 3)})
table = pd.DataFrame(rows)
print(table)
answer = float(table.loc[table["label"] == "AFIB", "PPV"].iloc[0])
teach.check_step("Step 1.4", "answer", globals())

**What the result shows.** With its threshold from fold 9, the AF alarm of the network is right in about 64 of 100 alarms. The thresholds of the rare labels lie far below 0.5, and their positive predictive values stay low: A label that 3 percent of the recordings carry produces many false alarms even with a good AUC.

### Step 1.5: The nine classes of the PhysioNet 2020 Challenge

**Goal.** Judge class by class a network trained for the nine classes of CPSC 2018, one of the databases of the PhysioNet 2020 Challenge. The training takes about a minute.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know how well a network recognises nine findings of the 12-lead ECG at once, from a normal ECG to ST elevation, and which findings it misses.  
> Data: The notebook holds a table named "data" with the 2,198 recordings of the test fold of PTB-XL. Each row is a recording: "record" gives its number. The nine columns "Normal", "AF", "I-AVB", "LBBB", "RBBB", "PAC", "PVC", "STD" and "STE" are 1 when the cardiologists gave the finding and 0 otherwise: a normal ECG, atrial fibrillation, first-degree atrioventricular block, left and right bundle branch block, premature atrial and ventricular contractions, ST depression and ST elevation, the nine classes of the China Physiological Signal Challenge 2018, a database of the PhysioNet 2020 Challenge. The nine columns "probability Normal" to "probability STE" hold the probabilities that a one-dimensional convolutional network, trained on folds 1 and 2, gave to the nine classes.  
> Task: For each of the nine classes, count the recordings with the class and compute the area under the ROC curve (AUC) of its probability against its column of 0 and 1. Draw the nine AUC values as a horizontal bar chart, sorted from the lowest to the highest, with a dashed line at 0.5.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print a table with the class, the number of recordings and the AUC, and keep the mean AUC of the nine classes, with three decimals, under the name "answer".  
> Test: Every AUC must lie between 0 and 1, and every class must have at least one recording in the table.  

In [ ]:
#@title Data of step 1.5: run this cell (it prepares the table "data" and shows it)
_labels9, _prob9 = _course_net9()
_i = np.where(te)[0]
data = pd.concat([pd.DataFrame({"record": ecgs["ecg_id"].to_numpy()[_i]}), _labels9.iloc[_i].reset_index(drop=True),
                  pd.DataFrame(np.round(_prob9, 4), columns=[f"probability {c}" for c in _labels9.columns])], axis=1)
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.5
teach.check_step("Step 1.5", "answer", globals())

In [ ]:
#@title Ready code of step 1.5 (run it only if the code of the assistant does not pass the check)
_labels9, _prob9 = _course_net9()
_i = np.where(te)[0]
data = pd.concat([pd.DataFrame({"record": ecgs["ecg_id"].to_numpy()[_i]}), _labels9.iloc[_i].reset_index(drop=True),
                  pd.DataFrame(np.round(_prob9, 4), columns=[f"probability {c}" for c in _labels9.columns])], axis=1)
names = ["Normal", "AF", "I-AVB", "LBBB", "RBBB", "PAC", "PVC", "STD", "STE"]
rows = []
for name in names:
    rows.append({"class": name, "recordings": int(data[name].sum()),
                     "AUC": round(roc_auc_score(data[name], data[f"probability {name}"]), 3)})
table = pd.DataFrame(rows).sort_values("AUC")
fig, ax = plt.subplots(figsize=(5, 3.4))
ax.barh(table["class"], table["AUC"], color="#1F5F8B")
ax.axvline(0.5, color="#999999", ls="--", lw=0.8)
ax.set_xlabel("AUC")
plt.show()
print(table.to_string(index=False))
answer = round(float(table["AUC"].mean()), 3)
teach.check_step("Step 1.5", "answer", globals())

**What the result shows.** The network recognises the bundle branch blocks almost perfectly and the premature atrial contractions poorly. ST elevation has only 3 recordings in the test fold and 5 in the training folds: The network has learned nothing about it, and an AUC measured on 3 recordings says nothing either. In the run of the course the mean AUC of the nine classes was about 0.848.

**Stop 1 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#last-two-hours-advanced-ecg-models-and-clinical-integration) and continue reading at «Last two hours: Advanced ECG models and clinical integration». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 2: ST elevation: the median beat, the J point and the alarm

*Lecture: [Practice stop 2](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#stop-2)* · *The same result on the Python path: Part 2, Sections 6 to 7*

The case of the week, record 13913, is read here as a STEMI alarm reads it. The single beats are aligned on their R peaks and their median gives one clean beat for each lead. The ST deviation at the J point is measured against the PR segment, the criteria of the universal definition decide the alarm, and the same alarm is then tried on the recordings of the test fold with left bundle branch block.

**At the end of this stop:** The number of beats in the median beat of the case (19), the number of leads that reach their cut-point (9) and the number of recordings with left bundle branch block of the test fold that raise the alarm (51 of 54).

In [ ]:
#@title Preparation of Stop 2: run this cell once (it loads the data of the stop; if Section 0 has not run in this session, it runs it first)
def _catch_up(upto, done, name="NB05_time_series_and_ecg_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
if "ecgs" not in globals():
    ecgs = ptbxl.table()
    signals = ptbxl.signals(ecgs)
    RHYTHMS = ["SR", "AFIB", "STACH", "SBRAD", "SARRH", "PVC"]
    labels = ptbxl.labels(ecgs, RHYTHMS)
    split = ecgs["split"].to_numpy()
    tr, va, te = split == "training", split == "validation", split == "test"

    r_peak_list = []
    for start in range(0, len(signals), 500):
        r_peak_list += beats.r_peaks_many(signals[start:start + 500])
    rr_list = [np.diff(p) / 100 for p in r_peak_list]
    cv = np.array([rr.std(ddof=1) / rr.mean() if len(rr) >= 3 else np.nan for rr in rr_list])


def _course_st():
    """ST deviation at the J point of the 12 leads of every recording of folds 9 and 10, on its median beat (computed once, about half a minute)."""
    if "_st_cache" in globals():
        return _st_cache
    rows = []
    for i in np.where(va | te)[0]:
        beat, n_beats = beats.median_beat(signals[i], peaks=r_peak_list[i])
        st = beats.st_deviation(beat)
        rows.append({"ecg_id": int(ecgs["ecg_id"].iloc[i]), "split": split[i], **st["st_mv"]})
    globals()["_st_cache"] = pd.DataFrame(rows)
    return _st_cache

CASE = 13913
_case = int(np.where(ecgs["ecg_id"] == CASE)[0][0])
print("Record", CASE, "·", ecgs["age"].iloc[_case], "years,", ecgs["sex"].iloc[_case], "·", ecgs["report"].iloc[_case][:60], "...")
print("Stop 2 is ready.")

### Step 2.1: The median beat of the case

**Goal.** Build from the single beats of the case one clean beat for each lead, as the software of an ECG machine does before it measures.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to build the median beat of a 12-lead ECG of 10 seconds, one clean beat for each lead, before the ST segment is measured.  
> Data: The notebook holds a table named "data" with record 13913 of PTB-XL, the 12-lead ECG of 10 seconds of a man of 68 years, recorded at 100 Hz. Each row is a sample, 10 ms after the one before: "time" gives its time in seconds, the twelve columns "I" to "V6" the values of the twelve leads in mV, and "R peak" is 1 at the samples of the R peaks and 0 elsewhere.  
> Task: Cut out every beat from 0.30 s before to 0.55 s after its R peak, and keep only the beats whose window lies entirely inside the 10 seconds. For each lead, take the median of these beats sample by sample: the median beat. Draw the median beat of V2 over the single beats of V2 in light grey, with the time in ms from the R peak on the horizontal axis.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of beats used, and keep it under the name "answer".  
> Test: Each beat must hold 86 samples, from −300 to 550 ms, and the median beat of V2 must lie inside the range of its single beats at every sample.  

In [ ]:
#@title Data of step 2.1: run this cell (it prepares the table "data" and shows it)
_peak = np.zeros(1000, dtype=int)
_peak[r_peak_list[_case]] = 1
data = pd.concat([pd.DataFrame({"time": np.round(np.arange(1000) / 100, 2)}),
                  pd.DataFrame(signals[_case].T, columns=list(beats.LEADS)), pd.DataFrame({"R peak": _peak})], axis=1)
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.1
teach.check_step("Step 2.1", "answer", globals())

In [ ]:
#@title Ready code of step 2.1 (run it only if the code of the assistant does not pass the check)
_peak = np.zeros(1000, dtype=int)
_peak[r_peak_list[_case]] = 1
data = pd.concat([pd.DataFrame({"time": np.round(np.arange(1000) / 100, 2)}),
                  pd.DataFrame(signals[_case].T, columns=list(beats.LEADS)), pd.DataFrame({"R peak": _peak})], axis=1)
peaks = np.where(data["R peak"].to_numpy() == 1)[0]
leads = data[["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]].to_numpy().T
beats_cut = np.stack([leads[:, p - 30:p + 56] for p in peaks if p - 30 >= 0 and p + 56 <= leads.shape[1]])
median_beat = np.median(beats_cut, axis=0)
ms = np.arange(-30, 56) * 10
fig, ax = plt.subplots(figsize=(5, 3.2))
for single in beats_cut[:, 7]:
    ax.plot(ms, single, color="#BBBBBB", lw=0.7)
ax.plot(ms, median_beat[7], color="#9B1B1B", lw=2, label="median beat of V2")
ax.axhline(0, color="#999999", lw=0.6)
ax.set_xlabel("time from the R peak (ms)")
ax.set_ylabel("mV")
ax.legend(fontsize=8)
plt.show()
answer = len(beats_cut)
print("Beats used:", answer)
teach.check_step("Step 2.1", "answer", globals())

**What the result shows.** The 19 beats of V2 lie almost on top of each other, so their median is a clean beat. The ST segment of V2 rises high above the baseline right after the QRS complex and merges with the T wave: the shape of an acute anterior infarction.

### Step 2.2: ST at the J point and the criteria of the universal definition

**Goal.** Measure the ST deviation of the twelve leads of the case and apply the criteria of the universal definition, as an automatic STEMI alarm does.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to measure the ST deviation at the J point in the twelve leads of a median beat and to see whether the ECG meets the ST criteria of the fourth universal definition of myocardial infarction.  
> Data: The notebook holds a table named "data" with the median beat of the 12 leads of record 13913 of PTB-XL, the ECG of a man of 68 years: the median of his single beats, sample by sample, aligned on their R peaks. Each row is a sample, 10 ms after the one before: "time" gives its time in ms from the R peak, from −300 to 550 ms, and the twelve columns "I" to "V6" the values of the twelve leads in mV. In this median beat, the QRS complex begins 70 ms before the R peak and ends, at the J point, 50 ms after it, as the slopes of all twelve leads show.  
> Task: For each lead, take as the reference level the mean of the values at 130, 120, 110 and 100 ms before the R peak, the 40 ms of the PR segment that end 20 ms before the QRS onset. The ST deviation of a lead is its value at the J point, 50 ms after the R peak, minus its reference level. A lead meets the criterion of the fourth universal definition of myocardial infarction when its ST elevation at the J point reaches its cut-point: 0.2 mV in V2 and V3 for men of 40 years or more, 0.25 mV for men under 40 and 0.15 mV for women, and 0.1 mV in every other lead. The alarm fires when at least two leads of the same group meet the criterion. The groups of contiguous leads are anterior (V1 to V6), inferior (II, III and aVF) and lateral (I and aVL); aVR belongs to no group.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the ST deviation of each lead in mV and in mm (1 mm = 0.1 mV), the leads that meet the criterion and the groups in which the alarm fires, and keep the number of leads that meet the criterion under the name "answer".  
> Test: The ST deviations of III and of aVR must be negative, because the report of the cardiologists describes ST depression in III.  

In [ ]:
#@title Data of step 2.2: run this cell (it prepares the table "data" and shows it)
_beat, _ = beats.median_beat(signals[_case], peaks=r_peak_list[_case])
data = pd.concat([pd.DataFrame({"time": np.arange(-30, 56) * 10}), pd.DataFrame(np.round(_beat.T, 3), columns=list(beats.LEADS))], axis=1)
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.2
teach.check_step("Step 2.2", "answer", globals())

In [ ]:
#@title Ready code of step 2.2 (run it only if the code of the assistant does not pass the check)
_beat, _ = beats.median_beat(signals[_case], peaks=r_peak_list[_case])
data = pd.concat([pd.DataFrame({"time": np.arange(-30, 56) * 10}), pd.DataFrame(np.round(_beat.T, 3), columns=list(beats.LEADS))], axis=1)
leads = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
t = data.set_index("time")
level = t.loc[[-130, -120, -110, -100], leads].mean()
st = (t.loc[50, leads] - level).round(3)
cut = {lead: (0.2 if lead in ("V2", "V3") else 0.1) for lead in leads}   # a man of 68 years
meet = [lead for lead in leads if st[lead] >= cut[lead]]
groups = {"anterior": ["V1", "V2", "V3", "V4", "V5", "V6"], "inferior": ["II", "III", "aVF"], "lateral": ["I", "aVL"]}
firing = [g for g, names in groups.items() if sum(lead in meet for lead in names) >= 2]
print(pd.DataFrame({"ST (mV)": st, "ST (mm)": (st * 10).round(1)}).T)
print("Leads that meet the criterion:", meet)
print("Alarm in the groups:", firing)
answer = len(meet)
teach.check_step("Step 2.2", "answer", globals())

**What the result shows.** 9 leads reach their cut-points, and the alarm fires in the anterior and the lateral group. V3 shows an elevation of about 10 mm, five times its cut-point. ST depression in III together with ST elevation in I and aVL is the reciprocal change that the report describes.

### Step 2.3: The alarm in left bundle branch block

**Goal.** Try the same alarm on all recordings of the test fold and see what it does in left bundle branch block, where the criteria do not apply.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to apply an automatic STEMI alarm, based on the ST criteria of the fourth universal definition of myocardial infarction, to all ECGs of a test set and to see how often it fires in complete left bundle branch block.  
> Data: The notebook holds a table named "data" with the 2,198 recordings of the test fold of PTB-XL. Each row is a recording: "record" gives its number, "sex" the sex of the patient ("M" or "F"), "age" the age in years, and "LBBB" is 1 when the cardiologists described a complete left bundle branch block and 0 otherwise. The twelve columns "I" to "V6" hold the ST deviation of the twelve leads in mV: the value at the J point of the median beat of the recording minus the level of its PR segment.  
> Task: A lead meets the criterion of the fourth universal definition of myocardial infarction when its ST elevation at the J point reaches its cut-point: 0.2 mV in V2 and V3 for men of 40 years or more, 0.25 mV for men under 40 and 0.15 mV for women, and 0.1 mV in every other lead. The alarm fires when at least two leads of the same group meet the criterion. The groups of contiguous leads are anterior (V1 to V6), inferior (II, III and aVF) and lateral (I and aVL); aVR belongs to no group. Apply the alarm to every recording, with the cut-points of V2 and V3 chosen by the sex and the age of its patient. Count the alarms among all recordings and among the recordings with left bundle branch block.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of alarms among all recordings and among the recordings with left bundle branch block, with the numbers of recordings, and keep the number of alarms among the recordings with left bundle branch block under the name "answer".  
> Test: There must be 2,198 recordings, 54 of them with left bundle branch block, and the alarms among them cannot be more than 54.  

In [ ]:
#@title Data of step 2.3: run this cell (it prepares the table "data" and shows it)
_st = _course_st()
_st = _st[_st["split"] == "test"].reset_index(drop=True)
_rows = ecgs.set_index("ecg_id").loc[_st["ecg_id"]]
data = pd.concat([pd.DataFrame({"record": _st["ecg_id"], "sex": np.where(_rows["sex"].to_numpy() == "M", "M", "F"),
                                "age": _rows["age"].to_numpy().astype(int),
                                "LBBB": _rows["codes"].apply(lambda d: int("CLBBB" in d)).to_numpy()}),
                  _st[list(beats.LEADS)].round(3)], axis=1)
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.3
teach.check_step("Step 2.3", "answer", globals())

In [ ]:
#@title Ready code of step 2.3 (run it only if the code of the assistant does not pass the check)
_st = _course_st()
_st = _st[_st["split"] == "test"].reset_index(drop=True)
_rows = ecgs.set_index("ecg_id").loc[_st["ecg_id"]]
data = pd.concat([pd.DataFrame({"record": _st["ecg_id"], "sex": np.where(_rows["sex"].to_numpy() == "M", "M", "F"),
                                "age": _rows["age"].to_numpy().astype(int),
                                "LBBB": _rows["codes"].apply(lambda d: int("CLBBB" in d)).to_numpy()}),
                  _st[list(beats.LEADS)].round(3)], axis=1)
leads = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
groups = [["V1", "V2", "V3", "V4", "V5", "V6"], ["II", "III", "aVF"], ["I", "aVL"]]


def alarm_of(row):
    v23 = 0.15 if row["sex"] == "F" else (0.2 if row["age"] >= 40 else 0.25)
    meet = {lead for lead in leads if row[lead] >= (v23 if lead in ("V2", "V3") else 0.1)}
    return any(sum(lead in meet for lead in g) >= 2 for g in groups)


alarm = data.apply(alarm_of, axis=1)
lbbb = data["LBBB"] == 1
print("All recordings:", int(alarm.sum()), "alarms in", len(data), "recordings")
print("Left bundle branch block:", int(alarm[lbbb].sum()), "alarms in", int(lbbb.sum()), "recordings")
answer = int(alarm[lbbb].sum())
teach.check_step("Step 2.3", "answer", globals())

**What the result shows.** 51 of the 54 recordings with left bundle branch block raise the alarm. In LBBB the ST segment is shifted opposite to the main deflection of the QRS complex, so V1 to V3 show ST elevation without any infarction. The universal definition therefore does not apply its ST criteria in LBBB, and an alarm has to recognise the block first, or use criteria made for it, such as those of Sgarbossa.

**Stop 2 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#qt-prolongation-and-its-risk) and continue reading at «QTc prolongation risk algorithm». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 3: Long-term monitoring: windows, AF burden and false alarms

*Lecture: [Practice stop 3](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#stop-3)* · *The same result on the Python path: Part 2, Sections 8 to 9*

A Holter recorder of the virtual clinic has followed patient VC-0128 for 24 hours. In this stop the beats of the recording are read in windows of 30 seconds, as the algorithm of a recorder does: The irregularity of the RR intervals in each window decides the alarm, the windows with an alarm give the AF burden, and the false alarms are traced back to their hours. The recording is synthetic, so the truth is known beat by beat, and nothing has to be downloaded.

**At the end of this stop:** The number of windows with an alarm (519 of 2,880), the estimated AF burden (percent 18.0, while the true burden is 15.4 percent) and the number of false alarms (75), with the hours in which they occur.

In [ ]:
#@title Preparation of Stop 3: run this cell once (it loads the data of the stop; if Section 0 has not run in this session, it runs it first)
def _catch_up(upto, done, name="NB05_time_series_and_ecg_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
if "holter_" not in globals():
    holter_ = holter.recording()
_b = holter_["detected_s"]
_af = np.zeros(len(_b), dtype=int)
for _s, _e in holter_["episodes"]:
    _af[(_b >= _s) & (_b <= _e)] = 1
beats_ = pd.DataFrame({"time": np.round(_b[1:], 3), "RR": np.round(np.diff(_b), 3), "AF": _af[1:]})
print(len(beats_), "beats in 24 hours")
print("Stop 3 is ready.")

### Step 3.1: The irregularity of the rhythm in windows of 30 seconds

**Goal.** Read the 24 hours as a monitor does, window by window, and count the windows whose RR intervals are irregular.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to read a 24-hour Holter recording in windows of 30 seconds and to see in how many windows the RR intervals are irregular enough for an alarm of atrial fibrillation (AF).  
> Data: The notebook holds a table named "data" with the beats of a 24-hour Holter recording of patient VC-0128 of the virtual clinic, a 76-year-old woman with palpitations, as the beat detector of the recorder found them. The recording starts at 08:00. Each row is a beat: "time" gives the time of the beat in seconds from the start of the recording, and "RR" the interval from the previous beat in seconds.  
> Task: Divide the 24 hours into windows of 30 seconds, starting at the beginning of the recording, and give each beat to the window in which its time falls. In every window that holds at least five RR intervals, compute the irregularity of the rhythm as the coefficient of variation: the standard deviation of its RR intervals, with the number of intervals minus one in the denominator, divided by their mean. Draw the coefficient of variation of every window against the clock time, with a horizontal line at 0.10.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of windows, and keep the number of windows whose coefficient of variation is above 0.10 under the name "answer".  
> Test: There must be 2,880 windows of 30 seconds, and the coefficients of variation must all be positive.  

In [ ]:
#@title Data of step 3.1: run this cell (it prepares the table "data" and shows it)
data = beats_[["time", "RR"]].rename(columns={"time": "time"})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.1
teach.check_step("Step 3.1", "answer", globals())

In [ ]:
#@title Ready code of step 3.1 (run it only if the code of the assistant does not pass the check)
data = beats_[["time", "RR"]].rename(columns={"time": "time"})
window = (data["time"] // 30).astype(int)
by_window = data.groupby(window)["RR"]
windows = pd.DataFrame({"n": by_window.count(), "cv": by_window.std() / by_window.mean()})
windows = windows[windows["n"] >= 5]
clock = (8 + windows.index * 30 / 3600) % 24
fig, ax = plt.subplots(figsize=(9, 3))
ax.scatter(windows.index * 30 / 3600, windows["cv"], s=3, color="#1F5F8B")
ax.axhline(0.10, color="#9B1B1B", ls="--", lw=1)
ax.set_xticks(range(0, 25, 4))
ax.set_xticklabels([f"{(8 + h) % 24:02d}:00" for h in range(0, 25, 4)])
ax.set_ylim(0, 0.5)
ax.set_xlabel("clock time")
ax.set_ylabel("coefficient of variation")
plt.show()
answer = int((windows["cv"] > 0.10).sum())
print(len(windows), "windows; above 0.10:", answer)
teach.check_step("Step 3.1", "answer", globals())

**What the result shows.** 519 of the 2,880 windows have a coefficient of variation above 0.10. Most of them lie in three blocks of the afternoon, the evening and the night, the episodes of AF; a few scattered windows and two short bursts of very high values lie elsewhere.

### Step 3.2: The AF burden

**Goal.** Turn the windows into the number that a Holter report gives: the share of the day spent in AF.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants the AF burden of a 24-hour Holter recording, the share of the recording spent in atrial fibrillation, as an algorithm that reads windows of 30 seconds would report it.  
> Data: The notebook holds a table named "data" with the beats of a 24-hour Holter recording of patient VC-0128 of the virtual clinic, a 76-year-old woman with palpitations, as the beat detector of the recorder found them. The recording starts at 08:00. Each row is a beat: "time" gives the time of the beat in seconds from the start of the recording, and "RR" the interval from the previous beat in seconds.  
> Task: Divide the 24 hours into windows of 30 seconds, starting at the beginning of the recording, and give each beat to the window in which its time falls. In every window that holds at least five RR intervals, compute the irregularity of the rhythm as the coefficient of variation: the standard deviation of its RR intervals, with the number of intervals minus one in the denominator, divided by their mean. Call a window AF when its coefficient of variation is above 0.10. The AF burden is the number of AF windows times 30 seconds, divided by the 24 hours of the recording. Also find the clock times of the first and the last AF window.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of AF windows, and keep the AF burden, in percent with one decimal, under the name "answer".  
> Test: The burden must equal the share of AF windows among all windows of 30 seconds, times 100.  

In [ ]:
#@title Data of step 3.2: run this cell (it prepares the table "data" and shows it)
data = beats_[["time", "RR"]].rename(columns={"time": "time"})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.2
teach.check_step("Step 3.2", "answer", globals())

In [ ]:
#@title Ready code of step 3.2 (run it only if the code of the assistant does not pass the check)
data = beats_[["time", "RR"]].rename(columns={"time": "time"})
window = (data["time"] // 30).astype(int)
by_window = data.groupby(window)["RR"]
windows = pd.DataFrame({"n": by_window.count(), "cv": by_window.std() / by_window.mean()})
windows = windows[windows["n"] >= 5]
af_windows = windows.index[windows["cv"] > 0.10]
answer = round(100 * len(af_windows) * 30 / (24 * 3600), 1)
first, last = [(8 + w * 30 / 3600) % 24 for w in (af_windows.min(), af_windows.max())]
print(len(af_windows), "AF windows; burden:", answer, "%")
print("first and last AF window:", f"{int(first):02d}:{int(first % 1 * 60):02d}", f"{int(last):02d}:{int(last % 1 * 60):02d}")
teach.check_step("Step 3.2", "answer", globals())

**What the result shows.** The windows give an AF burden of 18.0 percent, while the episodes confirmed by the cardiologist cover 15.4 percent of the day. The estimate is too high: Some windows outside the episodes raised an alarm as well. Note also that the first and the last AF window of the list are not the start and the end of one episode; the day held three episodes.

### Step 3.3: Where the false alarms come from

**Goal.** Compare the alarm with the confirmed episodes and find the hours of its false alarms, as the reader of a Holter recording would.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to compare the AF alarm of a 24-hour Holter recording with the episodes that a cardiologist confirmed, and to see at which hours of the day the false alarms occur.  
> Data: The notebook holds a table named "data" with the beats of a 24-hour Holter recording of patient VC-0128 of the virtual clinic, a 76-year-old woman with palpitations, as the beat detector of the recorder found them. The recording starts at 08:00. Each row is a beat: "time" gives the time of the beat in seconds from the start of the recording, and "RR" the interval from the previous beat in seconds. The column "AF" is 1 for the beats of the episodes of atrial fibrillation that a cardiologist confirmed on the whole recording, and 0 for all other beats.  
> Task: Divide the 24 hours into windows of 30 seconds, starting at the beginning of the recording, and give each beat to the window in which its time falls. In every window that holds at least five RR intervals, compute the irregularity of the rhythm as the coefficient of variation: the standard deviation of its RR intervals, with the number of intervals minus one in the denominator, divided by their mean. Call a window AF when its coefficient of variation is above 0.10. A window is a true AF window when more than half of its beats belong to a confirmed episode. Count the false alarms, the AF windows that are not true AF windows, and the missed windows, the true AF windows without an alarm. Then count the false alarms for each clock hour.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of missed windows and a small table of the false alarms by clock hour, and keep the total number of false alarms under the name "answer".  
> Test: The false alarms of all hours must add up to the total, and the AF windows must equal the true AF windows with an alarm plus the false alarms.  

In [ ]:
#@title Data of step 3.3: run this cell (it prepares the table "data" and shows it)
data = beats_.rename(columns={"time": "time"})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.3
teach.check_step("Step 3.3", "answer", globals())

In [ ]:
#@title Ready code of step 3.3 (run it only if the code of the assistant does not pass the check)
data = beats_.rename(columns={"time": "time"})
window = (data["time"] // 30).astype(int)
by_window = data.groupby(window)
windows = pd.DataFrame({"n": by_window["RR"].count(),
                        "cv": by_window["RR"].std() / by_window["RR"].mean(),
                        "true_af": by_window["AF"].mean() > 0.5})
windows = windows[windows["n"] >= 5]
alarm = windows["cv"] > 0.10
false_alarm = alarm & ~windows["true_af"]
missed = int((~alarm & windows["true_af"]).sum())
hour = ((8 + windows.index * 30 / 3600) % 24).astype(int)
by_hour = pd.Series(false_alarm.to_numpy(), index=pd.Index(hour, name="hour")).groupby(level=0).sum()
print("missed windows:", missed)
print(by_hour[by_hour > 0].rename("false alarms"))
answer = int(false_alarm.sum())
print("false alarms in all:", answer)
teach.check_step("Step 3.3", "answer", globals())

**What the result shows.** The alarm missed no window of AF, but it gave 75 false alarms, almost all of them in three hours. At 07:00 and at 23:00 the electrodes were noisy, and the detector of the recorder missed some beats and found others that were not there: The intervals scatter far more than in AF. Between 18:00 and 19:00 the patient had many premature atrial beats, each with a short and a long interval. A reader of the recording sees all three at a glance on the strip; an algorithm needs a quality rule and a rule for premature beats, which the animation on the lecture page and the Python path try.

**Stop 3 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#ecg-algorithms-with-regulatory-approval) and continue reading at «Overview of FDA/CE-approved ECG AI products». The weekly project can also be done on the prompt path: Its prompts are in the section «Weekly project on the prompt path» below.

## Weekly project on the prompt path

*Project of the week: [README of Week 5, Weekly project](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-05/README.md#weekly-project)*

The weekly project can also be built with the prompts below, one for each step. Run the preparation cell once: It loads the data of the project. Each step then works like a step of the stops, with one difference: The project has no check cell and no ready code, because its results go into the report. The test of each prompt must hold before the next prompt is sent, and every prompt goes into the AI usage log.

In [ ]:
#@title Preparation of the project: run this cell once (it loads the data of the project; if Section 0 has not run in this session, it runs it first)
def _catch_up(upto, done, name="NB05_time_series_and_ecg_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
if "ecgs" not in globals():
    ecgs = ptbxl.table()
    signals = ptbxl.signals(ecgs)
    RHYTHMS = ["SR", "AFIB", "STACH", "SBRAD", "SARRH", "PVC"]
    labels = ptbxl.labels(ecgs, RHYTHMS)
    split = ecgs["split"].to_numpy()
    tr, va, te = split == "training", split == "validation", split == "test"

    r_peak_list = []
    for start in range(0, len(signals), 500):
        r_peak_list += beats.r_peaks_many(signals[start:start + 500])
    rr_list = [np.diff(p) / 100 for p in r_peak_list]
    cv = np.array([rr.std(ddof=1) / rr.mean() if len(rr) >= 3 else np.nan for rr in rr_list])


def _course_st():
    """ST deviation at the J point of the 12 leads of every recording of folds 9 and 10, on its median beat (computed once, about half a minute)."""
    if "_st_cache" in globals():
        return _st_cache
    rows = []
    for i in np.where(va | te)[0]:
        beat, n_beats = beats.median_beat(signals[i], peaks=r_peak_list[i])
        st = beats.st_deviation(beat)
        rows.append({"ecg_id": int(ecgs["ecg_id"].iloc[i]), "split": split[i], **st["st_mv"]})
    globals()["_st_cache"] = pd.DataFrame(rows)
    return _st_cache

_codes = ecgs.set_index("ecg_id")["codes"]


def _project_table():
    _st = _course_st()
    _rows = ecgs.set_index("ecg_id").loc[_st["ecg_id"]]
    return pd.concat([pd.DataFrame({"record": _st["ecg_id"], "set": ["validation" if s == "validation" else "test" for s in _st["split"]],
                                    "sex": np.where(_rows["sex"].to_numpy() == "M", "M", "F"),
                                    "age": _rows["age"].to_numpy().astype(int),
                                    "acute infarction": _rows["acute_mi"].to_numpy().astype(int),
                                    "statements": [", ".join(sorted(d)) for d in _rows["codes"]]}),
                      _st[list(beats.LEADS)].round(3)], axis=1)
print("The project is ready.")

### Project step 1: The alarm against the acute infarctions

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to judge an automatic STEMI alarm, based on the ST criteria of the fourth universal definition of myocardial infarction, as a chest pain unit would: how many acute infarctions it finds and how many of its alarms are right.  
> Data: The notebook holds a table named "data" with the 4,381 recordings of folds 9 and 10 of PTB-XL. Each row is a recording: "record" gives its number, "set" its set ("validation" or "test"), "sex" the sex of the patient ("M" or "F") and "age" the age in years. "acute infarction" is 1 when the cardiologists classified an infarction of the recording as acute (stage I or I-II) and 0 otherwise, and "statements" lists the codes of all their statements, separated by commas, for example "CLBBB" for complete left bundle branch block, "LVH" for left ventricular hypertrophy and "PACE" for a pacemaker. The twelve columns "I" to "V6" hold the ST deviation of the twelve leads in mV: the value at the J point of the median beat of the recording minus the level of its PR segment.  
> Task: A lead meets the criterion of the fourth universal definition of myocardial infarction when its ST elevation at the J point reaches its cut-point: 0.2 mV in V2 and V3 for men of 40 years or more, 0.25 mV for men under 40 and 0.15 mV for women, and 0.1 mV in every other lead. The alarm fires when at least two leads of the same group meet the criterion. The groups of contiguous leads are anterior (V1 to V6), inferior (II, III and aVF) and lateral (I and aVL); aVR belongs to no group. Apply the alarm to every recording, with the cut-points of V2 and V3 chosen by the sex and the age of its patient. Compare the alarm with the column "acute infarction" and compute its sensitivity, its specificity and its positive predictive value, for all recordings and for each set.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table with the numbers of recordings, acute infarctions and alarms and the three values, one row for each set and one for all, and keep the sensitivity for all recordings, with three decimals, under the name "answer".  
> Test: There must be 4,381 recordings and 26 acute infarctions, and the true alarms cannot be more than 26.  

In [ ]:
#@title Data of project step 1: run this cell (it prepares the table "data" and shows it)
data = _project_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 2: Where the false alarms come from

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know which diagnoses the false alarms of an automatic STEMI alarm carry, to see where its criteria do not fit.  
> Data: The notebook holds a table named "data" with the 4,381 recordings of folds 9 and 10 of PTB-XL. Each row is a recording: "record" gives its number, "set" its set ("validation" or "test"), "sex" the sex of the patient ("M" or "F") and "age" the age in years. "acute infarction" is 1 when the cardiologists classified an infarction of the recording as acute (stage I or I-II) and 0 otherwise, and "statements" lists the codes of all their statements, separated by commas, for example "CLBBB" for complete left bundle branch block, "LVH" for left ventricular hypertrophy and "PACE" for a pacemaker. The twelve columns "I" to "V6" hold the ST deviation of the twelve leads in mV: the value at the J point of the median beat of the recording minus the level of its PR segment.  
> Task: A lead meets the criterion of the fourth universal definition of myocardial infarction when its ST elevation at the J point reaches its cut-point: 0.2 mV in V2 and V3 for men of 40 years or more, 0.25 mV for men under 40 and 0.15 mV for women, and 0.1 mV in every other lead. The alarm fires when at least two leads of the same group meet the criterion. The groups of contiguous leads are anterior (V1 to V6), inferior (II, III and aVF) and lateral (I and aVL); aVR belongs to no group. Apply the alarm to every recording, with the cut-points of V2 and V3 chosen by the sex and the age of its patient. A false alarm is an alarm in a recording without an acute infarction. Count, for every statement, the false alarms whose recording carries it, and for the statements "CLBBB", "PACE" and "LVH" also the share of their recordings that raise the alarm.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the 15 statements with the most false alarms, and keep the number of false alarms in recordings with "CLBBB" under the name "answer".  
> Test: The false alarms of a statement cannot be more than the recordings that carry it, and the total of false alarms must equal the alarms minus the true alarms of step 1.  

In [ ]:
#@title Data of project step 2: run this cell (it prepares the table "data" and shows it)
data = _project_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 3: A sweep of the cut-points

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to see how the sensitivity and the positive predictive value of an automatic STEMI alarm change when all its cut-points are raised together.  
> Data: The notebook holds a table named "data" with the 4,381 recordings of folds 9 and 10 of PTB-XL. Each row is a recording: "record" gives its number, "set" its set ("validation" or "test"), "sex" the sex of the patient ("M" or "F") and "age" the age in years. "acute infarction" is 1 when the cardiologists classified an infarction of the recording as acute (stage I or I-II) and 0 otherwise, and "statements" lists the codes of all their statements, separated by commas, for example "CLBBB" for complete left bundle branch block, "LVH" for left ventricular hypertrophy and "PACE" for a pacemaker. The twelve columns "I" to "V6" hold the ST deviation of the twelve leads in mV: the value at the J point of the median beat of the recording minus the level of its PR segment.  
> Task: A lead meets the criterion of the fourth universal definition of myocardial infarction when its ST elevation at the J point reaches its cut-point: 0.2 mV in V2 and V3 for men of 40 years or more, 0.25 mV for men under 40 and 0.15 mV for women, and 0.1 mV in every other lead. The alarm fires when at least two leads of the same group meet the criterion. The groups of contiguous leads are anterior (V1 to V6), inferior (II, III and aVF) and lateral (I and aVL); aVR belongs to no group. Apply the alarm to every recording, with the cut-points of V2 and V3 chosen by the sex and the age of its patient. Repeat the alarm with every cut-point raised by 0, 0.05, 0.10, 0.15 and 0.20 mV. For each margin, compute the sensitivity, the specificity and the positive predictive value against the column "acute infarction", and draw the sensitivity and the positive predictive value against the margin.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table with one row for each margin, and keep the sensitivity with a margin of 0.10 mV, with three decimals, under the name "answer".  
> Test: The number of alarms cannot rise when the margin grows, and the row with a margin of 0 must equal the result of step 1.  

In [ ]:
#@title Data of project step 3: run this cell (it prepares the table "data" and shows it)
data = _project_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 4: Leaving out the recordings where the criteria do not apply

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to judge an automatic STEMI alarm only on the ECGs to which the ST criteria of the universal definition apply, leaving out left bundle branch block and paced rhythms.  
> Data: The notebook holds a table named "data" with the 4,381 recordings of folds 9 and 10 of PTB-XL. Each row is a recording: "record" gives its number, "set" its set ("validation" or "test"), "sex" the sex of the patient ("M" or "F") and "age" the age in years. "acute infarction" is 1 when the cardiologists classified an infarction of the recording as acute (stage I or I-II) and 0 otherwise, and "statements" lists the codes of all their statements, separated by commas, for example "CLBBB" for complete left bundle branch block, "LVH" for left ventricular hypertrophy and "PACE" for a pacemaker. The twelve columns "I" to "V6" hold the ST deviation of the twelve leads in mV: the value at the J point of the median beat of the recording minus the level of its PR segment.  
> Task: Leave out every recording whose statements include "CLBBB" or "PACE". A lead meets the criterion of the fourth universal definition of myocardial infarction when its ST elevation at the J point reaches its cut-point: 0.2 mV in V2 and V3 for men of 40 years or more, 0.25 mV for men under 40 and 0.15 mV for women, and 0.1 mV in every other lead. The alarm fires when at least two leads of the same group meet the criterion. The groups of contiguous leads are anterior (V1 to V6), inferior (II, III and aVF) and lateral (I and aVL); aVR belongs to no group. Apply the alarm to every recording, with the cut-points of V2 and V3 chosen by the sex and the age of its patient. Compute the sensitivity, the specificity and the positive predictive value against the column "acute infarction" on the recordings that remain, and compare them with step 1.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of recordings left out and the three values, and keep the positive predictive value, with three decimals, under the name "answer".  
> Test: The recordings left out and those that remain must add up to 4,381.  

In [ ]:
#@title Data of project step 4: run this cell (it prepares the table "data" and shows it)
data = _project_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

<a id="part-2"></a>

# Part 2: Python path

**For whom?** This part is for participants who want to learn to write Python. Part 1 is not needed for it: After Section 0, start here.

**How it works.** Run the cells from top to bottom with Shift+Enter. The *Python step* at the start teaches the parts of the language that the week needs. In the numbered sections, exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells: Open them only after trying. *Observe and reflect* cells ask open questions and have no solution cell. *Python note* cells explain the language where a new construct first appears. Cells marked *Advanced (optional)* are extensions for those who already program. A participant who opens this part in the middle runs the *Catch-up* cell at the start of that section. Some exercise cells begin with a line that prepares a table named `data`: It serves the prompt of the exercise in the prompt workshop of the interactive lab, and on the Python path the exercise can be solved with the tables named in its text.

## Python step 5: Signals in batches

This week the data are signals: twelve leads of numbers in their order, and thousands of recordings of them. This step works with the synthetic ECGs of the virtual clinic. It turns one ECG into an array of leads × samples, cuts it into windows, stacks the ECGs of 40 patients into one array with several labels for each patient, and goes through them in batches, as a network does.

In [ ]:
#@title Preparation of the Python step: run this cell once (if Section 0 has not run in this session, it runs it)
def _catch_up(upto, done, name="NB05_time_series_and_ecg_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
print("The Python step is ready.")

In [ ]:
# Python practice: a small example for learning the language
cohort = vclinic.make_cohort(500)
ecg = vclinic.patient_ecg(cohort, "VC-0128", fs=100)
x = ecg.leads.to_numpy().T
print(type(x).__name__, x.shape, x.dtype)
print("Lead II, first five samples (mV):", x[1, :5].round(3))
print("The first second of all leads:", x[:, :100].shape)

**Python note.** `ecg.leads` is a table with one column per lead and one row per sample. `.to_numpy()` turns it into an array of numbers, and `.T` *transposes* it, so that rows and columns change places: The first index now chooses the lead and the second the sample, the order that PTB-XL and the networks of the week use. Python counts from 0, so `x[1]` is lead II. `x[:, :100]` means all leads (`:`) and the samples from the start up to, but not including, sample 100: the first second at 100 Hz.

In [ ]:
# Python practice: cutting a recording into windows
fs = 100
length = 2 * fs                       # windows of 2 seconds
starts = list(range(0, x.shape[1] - length + 1, length))
windows = np.stack([x[:, s:s + length] for s in starts])
print(starts)
print(windows.shape)                  # (windows, leads, samples)

**Python note.** `range(start, stop, step)` counts from `start` in steps of `step` and stops before `stop`; here it gives the first sample of every window. `[x[:, s:s + length] for s in starts]` is a *list comprehension*: It builds a list with one window for every value of `s`. `np.stack` puts the windows on top of each other along a new first axis, so the five windows of 12 leads × 200 samples become one array of shape (5, 12, 200).

In [ ]:
# Python practice: many recordings in one array, with several labels each
first = cohort.iloc[:40]
X = np.stack([vclinic.patient_ecg(cohort, p, fs=100).leads.to_numpy().T for p in first["patient_id"]])
labels_vc = pd.DataFrame({"AF": (first["ecg_rhythm"] == "AF").astype(int),
                          "LBBB": (first["ecg_conduction"] == "LBBB").astype(int),
                          "STEMI": (first["stemi_territory"] != "none").astype(int)})
labels_vc.index = first["patient_id"]
print(X.shape)
print(labels_vc.sum())
print(labels_vc[labels_vc.sum(axis=1) > 0])

**Python note.** `(first["ecg_rhythm"] == "AF")` compares every patient with "AF" and gives `True` or `False`; `.astype(int)` turns them into 1 and 0. A table with one column of 0 and 1 for each label is called *multi-hot*: A patient can have 1 in several columns at once. `labels_vc.sum()` adds up each column, the number of patients with each label; `labels_vc.sum(axis=1)` adds up each row, the number of labels of each patient.

In [ ]:
# Python practice: going through the recordings in batches
batch = 16
for start in range(0, len(X), batch):
    part = X[start:start + batch]
    print(start, part.shape, "largest value of lead II:", round(float(part[:, 1].max()), 2))

**Python note.** A network reads its recordings in *batches*, a few at a time, because thousands of recordings do not fit into one step of the computation. `X[start:start + batch]` takes 16 recordings; the last batch holds the 8 that remain, because a slice may end beyond the last element. `part[:, 1]` is lead II of every recording of the batch.

**Quick check.** An array of ECGs has the shape (40, 12, 1000). What does `X[5, 1]` hold?

<details><summary>Answer</summary>

Lead II of the sixth recording, 1,000 numbers: The first index chooses the recording and the second the lead, both counted from 0.

</details>

**Exercise 5.1.** Into how many windows of 2.5 seconds, 250 samples, without overlap does the recording `x` fall? Build them as in `starts`.

In [ ]:
data = exercise_table("5.1")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 5.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = len(range(0, x.shape[1] - 250 + 1, 250))
teach.check("Exercise 5.1", answer)

**Exercise 5.2.** How many of the 40 patients in `labels_vc` carry at least two of the three labels?

In [ ]:
data = exercise_table("5.2")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 5.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int((labels_vc.sum(axis=1) >= 2).sum())
teach.check("Exercise 5.2", answer)

**Exercise 5.3.** Go through `X` in batches of 16 and collect the largest value of lead II of each of the 40 recordings. What is the mean of these 40 values in mV? Round to three decimals.

In [ ]:
data = exercise_table("5.3")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 5.3", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
largest = []
for start in range(0, len(X), 16):
    largest += list(X[start:start + 16, 1].max(axis=1))
answer = round(float(np.mean(largest)), 3)
teach.check("Exercise 5.3", answer)

## 1. The 12-lead ECGs of PTB-XL

*Lecture: [Modelling the ECG as a time series](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#the-ecg-as-a-time-series)*

PTB-XL holds 21,799 clinical 12-lead ECGs of 10 seconds, divided by patient into ten folds. The course uses four of them: folds 1 and 2 for training, fold 9 for validation and fold 10 for the test. `ptbxl.table()` gives one row per recording, with the patient, the age and the sex, the fold and the statements of the cardiologists.

**From the formulas to the code.** $N = f_s \times T = 100 \times 10 = 1000$ samples per lead: `signals.shape` is (8737, 12, 1000). `ptbxl.signals(ecgs)` reads all recordings into one array, in batches of 500, and keeps a copy on the disk, so that the second run takes a second. `ptbxl.labels(ecgs, RHYTHMS)` builds the multi-hot table of the six labels of the course.

**Started here?** Run the catch-up cell below once. It runs the code that Section 1 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB05_time_series_and_ecg_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(1, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
ecgs = ptbxl.table()
signals = ptbxl.signals(ecgs)
RHYTHMS = ["SR", "AFIB", "STACH", "SBRAD", "SARRH", "PVC"]
labels = ptbxl.labels(ecgs, RHYTHMS)
split = ecgs["split"].to_numpy()
tr, va, te = split == "training", split == "validation", split == "test"

print(signals.shape, signals.dtype)
print(ecgs.groupby("split").agg(recordings=("ecg_id", "size"), patients=("patient_id", "nunique")))

**Python note.** `ecgs` holds one row per recording. The column `codes` keeps the statements of a recording as a *dictionary*, pairs of a code and a number, for example `{"NORM": 100.0, "SR": 0.0}`: The number is the likelihood that the cardiologists gave to a diagnostic statement, and 0 for statements of rhythm and form. `ecgs.groupby("split").agg(...)` groups the rows by set and computes two columns at once: the number of recordings and the number of different patients (`nunique`).

In [ ]:
print(labels.groupby(split).sum().T)
patients_of = {s: set(ecgs.loc[split == s, "patient_id"]) for s in ("training", "validation", "test")}
print("Patients in two sets:", len(patients_of["training"] & patients_of["test"]) + len(patients_of["training"] & patients_of["validation"]) + len(patients_of["validation"] & patients_of["test"]))

**Python note.** `labels.groupby(split).sum()` counts the recordings with each label in each set, and `.T` turns the table so that the labels become rows. `set(...)` keeps every patient once, and `&` gives the patients that two sets share. The answer 0 shows the rule of the folds of PTB-XL: All recordings of one patient lie in one fold.

In [ ]:
# The case of the week: record 13913 of the test fold
CASE = 13913
case = int(np.where(ecgs["ecg_id"] == CASE)[0][0])
print(ecgs.loc[case, ["ecg_id", "age", "sex", "split", "codes"]].to_dict())
print(ecgs.loc[case, "report"])
case_leads = pd.DataFrame(signals[case].T, columns=list(beats.LEADS))
teach.plot_12lead(np.arange(1000) / 100, case_leads, title="PTB-XL, record 13913 (CC BY 4.0)")
plt.show()

**Observe and reflect.** Read the 12 leads of the case before any program does. Which rhythm do you see, and in which leads is the ST segment elevated or depressed? Keep your answer: Sections 4 and 6 compare it with the network and with the alarm of the course.

**Exercise A.** How many recordings of the test fold carry both sinus rhythm (SR) and ventricular premature complexes (PVC)?

In [ ]:
data = exercise_table("A")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise A", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int(((labels["SR"].to_numpy() == 1) & (labels["PVC"].to_numpy() == 1) & te).sum())
teach.check("Exercise A", answer)

## 2. RR intervals and a rule for AF

*Lecture: [A rule first: RR intervals and atrial fibrillation](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#a-rule-first-rr-intervals-and-atrial-fibrillation)*

The rule of the lecture calls a recording AF when the coefficient of variation of its RR intervals is above 0.10. The R peaks come from the method of Week 1, the Pan-Tompkins signal, applied to all twelve leads at once: A lead with a small QRS complex counts as much as one with a large complex, so one flat lead does not hide a beat.

**From the formulas to the code.** $RR_i = (r_{i+1} - r_i)/f_s$ with the R peaks $r_i$ in samples: `np.diff(p) / 100`. $\mathrm{CV} = s_{RR}/\overline{RR}$, with $m - 1$ in the denominator of $s_{RR}$: `rr.std(ddof=1) / rr.mean()`. `beats.r_peaks_many(signals[start:start + 500])` finds the R peaks of 500 recordings at once.

**Started here?** Run the catch-up cell below once. It runs the code that Section 2 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB05_time_series_and_ecg_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(2, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
r_peak_list = []
for start in range(0, len(signals), 500):
    r_peak_list += beats.r_peaks_many(signals[start:start + 500])
rr_list = [np.diff(p) / 100 for p in r_peak_list]
cv = np.array([rr.std(ddof=1) / rr.mean() if len(rr) >= 3 else np.nan for rr in rr_list])

print(len(r_peak_list), "recordings;", "RR intervals per recording (median):", int(np.median([len(r) for r in rr_list])))

**Python note.** `r_peak_list += ...` adds the R peaks of every batch to the end of the list. `np.diff(p)` gives the differences between neighbouring R peaks in samples, and dividing by 100 turns them into seconds. `ddof=1` puts $m - 1$ instead of $m$ in the denominator of the standard deviation, as in the formula of the lecture. `if len(rr) >= 3 else np.nan` writes *not a number* for a recording with fewer than three intervals, so that it cannot pass as regular.

In [ ]:
# RR intervals of four recordings of the test fold
fig, axes = plt.subplots(1, 4, figsize=(12, 2.6), sharey=True)
for ax, record in zip(axes, (9, 318, 76, 452)):
    i = int(np.where(ecgs["ecg_id"] == record)[0][0])
    labels_i = [c for c in RHYTHMS if labels.iloc[i][c] == 1]
    ax.plot(rr_list[i] * 1000, marker="o", ms=3)
    ax.set_title(f"{record}: {', '.join(labels_i)} · CV {cv[i]:.3f}", fontsize=9)
    ax.set_xlabel("interval")
axes[0].set_ylabel("RR (ms)")
plt.show()

In [ ]:
def macro_auc(y_true, prob):
    """Mean of the AUCs of the labels: one AUC per column."""
    return float(np.mean([roc_auc_score(y_true[:, k], prob[:, k]) for k in range(y_true.shape[1])]))


def youden_threshold(y_true, prob):
    """The threshold at which sensitivity + specificity is highest."""
    fpr, tpr, thresholds = roc_curve(y_true, prob)
    return float(thresholds[np.argmax(tpr - fpr)])


def se_sp_ppv(y_true, alarm):
    """Sensitivity, specificity and positive predictive value of a yes-or-no decision."""
    y_true, alarm = np.asarray(y_true, dtype=bool), np.asarray(alarm, dtype=bool)
    tp, fp = int((alarm & y_true).sum()), int((alarm & ~y_true).sum())
    return tp / y_true.sum(), 1 - fp / (~y_true).sum(), tp / max(1, tp + fp)

af = labels["AFIB"].to_numpy() == 1
has_cv = np.isfinite(cv)
for name, m in (("validation", va), ("test", te)):
    m = m & has_cv
    se, sp, ppv = se_sp_ppv(af[m], cv[m] > 0.10)
    print(f"{name}: AUC {roc_auc_score(af[m], cv[m]):.3f} · sensitivity {se:.3f} · specificity {sp:.3f} · PPV {ppv:.3f}")
false_alarm = te & has_cv & (cv > 0.10) & ~af
print("False alarms on the test fold:", int(false_alarm.sum()), labels[false_alarm].sum().to_dict())

**Python note.** `se_sp_ppv` counts the true and false alarms of a decision of yes or no and returns three numbers, which `se, sp, ppv = ...` unpacks. `roc_auc_score(af[m], cv[m])` uses the CV itself, not the decision, and gives the AUC over all thresholds. `labels[false_alarm].sum()` counts the labels of the recordings with a false alarm; one recording can add to several of them.

**Observe and reflect.** Most false alarms of the rule fall on recordings with premature beats or with sinus arrhythmia. Which second criterion, one that a cardiologist uses on the strip, would you add to the rule, and which part of the signal would it need?

**Exercise B.** What is the sensitivity of the rule on the test fold when the threshold is raised from 0.10 to 0.15? Round to three decimals.

In [ ]:
data = exercise_table("B")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise B", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
m = te & has_cv
answer = round(se_sp_ppv(af[m], cv[m] > 0.15)[0], 3)
teach.check("Exercise B", answer)

## 3. A one-dimensional convolutional network

*Lecture: [One-dimensional convolution: A kernel that slides along time](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#one-dimensional-convolution-a-kernel-that-slides-along-time)*

A kernel of $k$ samples slides along the 12 leads, multiplies the values under it by its weights and writes their sum into its feature map. The network of the course has four such layers with stride 2, so that the length of the signal halves at every layer, from 1,000 to 63 samples, while the number of channels grows from 12 to 128. The mean of every channel over time feeds the last layer, which gives one score per label.

**From the formulas to the code.** $y[n] = b + \sum_{c}\sum_{j} w_{c,j}\, x_c[n + j]$: `nn.Conv1d(12, 32, 7, stride=2, padding=3)`. $P_{conv} = C_{in} \times C_{out} \times k + C_{out}$: `sum(p.numel() for p in layer.parameters())`. The mean over time is `.mean(dim=2)`, and the loss of the six labels is `nn.BCEWithLogitsLoss()`, the binary cross-entropy computed from the scores before the sigmoid.

**Started here?** Run the catch-up cell below once. It runs the code that Section 3 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB05_time_series_and_ecg_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(3, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
# The slope kernel of the lecture on lead II of the case, by hand
lead_ii = signals[case, 1, :300]
kernel = np.array([-1.0, 0.0, 1.0])
feature_map = np.array([np.sum(kernel * lead_ii[n - 1:n + 2]) for n in range(1, len(lead_ii) - 1)])
n_max = int(np.argmax(feature_map)) + 1
print("Largest output at sample", n_max, "·", lead_ii[n_max - 1:n_max + 2].round(2), "→", round(float(feature_map.max()), 2), "mV")
fig, (a, b) = plt.subplots(2, 1, figsize=(9, 3.6), sharex=True)
a.plot(np.arange(300) / 100, lead_ii)
a.set_ylabel("lead II (mV)")
b.plot(np.arange(1, 299) / 100, feature_map, color="#9B1B1B")
b.set_ylabel("feature map")
b.set_xlabel("time (s)")
plt.show()

**Python note.** `lead_ii[n - 1:n + 2]` takes three samples around sample `n`, and `np.sum(kernel * ...)` multiplies them by the three weights and adds the products: one value of the feature map. The list comprehension repeats this at every position. The largest output lies on the steepest upstroke of a QRS complex. A convolutional layer does the same with 32 kernels at once, each with weights that it learns.

In [ ]:
def conv_block(c_in, c_out, k):
    """One layer: a convolution along time with stride 2, batch normalisation and ReLU."""
    return [nn.Conv1d(c_in, c_out, k, stride=2, padding=k // 2), nn.BatchNorm1d(c_out), nn.ReLU()]


class RhythmNet(nn.Module):
    """Four convolutions over the 12 leads, the mean of every channel over time and one output per label."""

    def __init__(self, n_labels=6):
        super().__init__()
        self.features = nn.Sequential(*conv_block(12, 32, 7), *conv_block(32, 64, 5),
                                      *conv_block(64, 64, 5), *conv_block(64, 128, 3))
        self.out = nn.Linear(128, n_labels)

    def forward(self, x):
        return self.out(self.features(x).mean(dim=2))


def predict(net, *inputs, batch=256):
    """The probabilities of the labels, computed in batches without training."""
    net.eval()
    with torch.no_grad():
        return torch.cat([torch.sigmoid(net(*[x[i:i + batch] for x in inputs]))
                          for i in range(0, len(inputs[0]), batch)]).numpy()

rhythm_net = RhythmNet(len(RHYTHMS))
print("Trainable parameters:", sum(p.numel() for p in rhythm_net.parameters()))
h = torch.zeros(1, 12, 1000)
for layer in rhythm_net.features:
    h = layer(h)
    if isinstance(layer, nn.Conv1d):
        print(f"{layer.in_channels:3d} → {layer.out_channels:3d} channels, kernel {layer.kernel_size[0]} · output {tuple(h.shape)}")

**Python note.** `class RhythmNet(nn.Module)` follows the pattern of the networks of Weeks 3 and 4: `__init__` creates the layers and `forward` says how a batch passes through them. `conv_block` returns the three layers of one level as a list, and `*` unpacks the four lists into `nn.Sequential`. `padding=k // 2` adds half a kernel of zeros at each end, so that stride 2 gives exactly half the length. `predict` switches the network to evaluation with `net.eval()` and computes the probabilities in batches without gradients.

In [ ]:
x_all = torch.tensor(np.clip(signals, -5, 5))
y_all = torch.tensor(np.array(labels, dtype=np.float32))
x_train, y_train = x_all[tr], y_all[tr]
x_val, x_test = x_all[va], x_all[te]
y_val, y_test = labels.to_numpy()[va], labels.to_numpy()[te]

torch.manual_seed(RANDOM_STATE)
rhythm_net = RhythmNet(len(RHYTHMS))
EPOCHS, BATCH = 15, 64
optimizer = torch.optim.Adam(rhythm_net.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.003, total_steps=EPOCHS * math.ceil(len(x_train) / BATCH))
loss_fn = nn.BCEWithLogitsLoss()
best_auc, best_epoch, val_history = -1.0, 0, []
for epoch in range(1, EPOCHS + 1):
    rhythm_net.train()
    order = torch.randperm(len(x_train))
    for i in range(0, len(order), BATCH):
        b = order[i:i + BATCH]
        optimizer.zero_grad()
        loss = loss_fn(rhythm_net(x_train[b]), y_train[b])
        loss.backward()
        optimizer.step()
        scheduler.step()
    val_auc = macro_auc(y_val, predict(rhythm_net, x_val))
    val_history.append(val_auc)
    if val_auc > best_auc:
        best_auc, best_epoch, best_state = val_auc, epoch, copy.deepcopy(rhythm_net.state_dict())
    print(f"epoch {epoch:2d} · mean AUC on fold 9: {val_auc:.3f}")
rhythm_net.load_state_dict(best_state)
val_prob, test_prob = predict(rhythm_net, x_val), predict(rhythm_net, x_test)

fig, ax = plt.subplots(figsize=(5, 2.8))
ax.plot(range(1, EPOCHS + 1), val_history, marker="o")
ax.set_xlabel("epoch")
ax.set_ylabel("mean AUC, fold 9")
plt.show()
print("Best epoch:", best_epoch, "· mean AUC on fold 9:", round(best_auc, 3))

**Python note.** The loop is that of Weeks 3 and 4. `torch.randperm` shuffles the training recordings in every epoch, `OneCycleLR` raises the learning rate to 0.003 and lowers it again over the 15 epochs, and the best epoch is chosen by the mean AUC on fold 9. Training takes about a minute on a laptop; in the run of the course the best epoch was 12, with a mean AUC of 0.881 on fold 9.

**Exercise C.** How many parameters does the same network have for twelve labels instead of six, `RhythmNet(12)`?

In [ ]:
answer = None   # your answer
teach.check("Exercise C", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = sum(p.numel() for p in RhythmNet(12).parameters())
teach.check("Exercise C", answer)

**AF, VT and SVT: what the network must tell apart.** *Lecture: [AF, VT and SVT on the ECG and in the network](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#af-vt-and-svt-on-the-ecg-and-in-the-network)*

PTB-XL holds no recording of ventricular tachycardia (VT) and only a few of supraventricular tachycardia (SVT). The cells below therefore make 1,200 synthetic 12-lead ECGs with the generator of the virtual clinic, 300 for each of four rhythms: sinus rhythm, atrial fibrillation (AF), SVT and VT. The same network learns the four rhythms from 800 of them and is tested on the other 400. The last cell shows what the network learned from synthetic signals alone, on the real recordings of PTB-XL.

**From the formulas to the code.** One rhythm per recording, so the four scores pass through the softmax function, $p_k = e^{z_k}/\sum_j e^{z_j}$, and the loss is the cross-entropy of Weeks 3 and 4: `nn.functional.cross_entropy(scores, rhythm)`. The parameters of each rhythm follow the lecture: an irregular RR interval for AF, a regular rate of 150 to 220 per minute with a narrow QRS for SVT, and a regular rate of 130 to 200 per minute with a QRS of 140 to 180 ms for VT.

In [ ]:
from vclinic import simulate

KINDS = ["sinus", "AF", "SVT", "VT"]
KIND_NAMES = ["sinus", "AF", "SVT", "VT"]


def synthetic_rhythm(kind, seed):
    """A synthetic 12-lead ECG of 10 s at 100 Hz: sinus rhythm, AF, SVT or VT."""
    r = np.random.default_rng(seed)
    if kind == "sinus":
        p = dict(rhythm="sinus", hr_bpm=r.uniform(55, 95))
    elif kind == "AF":
        p = dict(rhythm="AF", hr_bpm=r.uniform(80, 140))
    elif kind == "SVT":   # regular, narrow QRS, no visible P wave
        p = dict(rhythm="SVT", hr_bpm=r.uniform(150, 220), qrs_ms=r.uniform(75, 105), hrv=0.05)
    else:                 # regular, wide QRS, T wave opposite to the QRS
        p = dict(rhythm="VT", conduction="LBBB", hr_bpm=r.uniform(130, 200), qrs_ms=r.uniform(140, 180),
                 axis_deg=r.uniform(-90, 0), hrv=0.05)
    p["qt_ms"] = 400 * np.sqrt(60 / p["hr_bpm"])   # the QT shortens as the rate rises
    p.setdefault("axis_deg", r.uniform(-10, 90))
    p["r_amp"] = r.uniform(0.9, 1.8)
    ecg = simulate(p, duration_s=10, fs=100, noise=str(r.choice(["low", "clinic"])), seed=seed)
    return ecg.leads.to_numpy().T.astype(np.float32)


synth_x = np.stack([synthetic_rhythm(kind, 10000 * (k + 1) + i) for k, kind in enumerate(KINDS) for i in range(300)])
synth_y = np.repeat(np.arange(len(KINDS)), 300)
synth_order = np.random.default_rng(0).permutation(len(synth_y))
synth_train, synth_test = synth_order[:800], synth_order[800:]

torch.manual_seed(RANDOM_STATE)
synth_net = RhythmNet(len(KINDS))
synth_optimizer = torch.optim.Adam(synth_net.parameters(), lr=0.003)
synth_tensor, synth_target = torch.tensor(np.clip(synth_x, -5, 5)), torch.tensor(synth_y)
for synth_epoch in range(5):
    synth_net.train()
    synth_perm = synth_train[torch.randperm(len(synth_train)).numpy()]
    for start in range(0, len(synth_perm), 32):
        batch_index = synth_perm[start:start + 32]
        synth_optimizer.zero_grad()
        synth_loss = nn.functional.cross_entropy(synth_net(synth_tensor[batch_index]), synth_target[batch_index])
        synth_loss.backward()
        synth_optimizer.step()
synth_net.eval()
with torch.no_grad():
    synth_pred = synth_net(synth_tensor[synth_test]).argmax(dim=1).numpy()

print("Synthetic recordings:", synth_x.shape, "· training", len(synth_train), "· test", len(synth_test))
print(pd.crosstab(pd.Series(np.array(KIND_NAMES)[synth_y[synth_test]], name="true rhythm"),
                  pd.Series(np.array(KIND_NAMES)[synth_pred], name="network")))
fig, axes = plt.subplots(4, 1, figsize=(9, 6), sharex=True)
for panel, kind in enumerate(KINDS):
    first_of_kind = synth_test[synth_y[synth_test] == panel][0]
    axes[panel].plot(np.arange(400) / 100, synth_x[first_of_kind, 1, :400], lw=1)
    axes[panel].set_ylabel(KIND_NAMES[panel])
axes[-1].set_xlabel("time (s), lead II")
plt.show()

**Python note.** `synthetic_rhythm(kind, seed)` chooses the parameters of one patient at random within the range of the rhythm and passes them to `simulate`, the generator of Week 1; the same seed always gives the same ECG. `np.repeat(np.arange(4), 300)` writes the label of each recording: 300 zeros, 300 ones, and so on. `synth_net(...).argmax(dim=1)` takes the rhythm with the largest score. `pd.crosstab(true, predicted)` counts every pair, so the correct answers lie on the diagonal.

In [ ]:
has_code = lambda code: ecgs["codes"].apply(lambda d: code in d).to_numpy()
real_kind = np.select([has_code("SVTAC") | has_code("PSVT"), has_code("AFIB"), has_code("NORM") & has_code("SR")],
                      ["SVT", "AF", "sinus"], "other")
real = real_kind != "other"
with torch.no_grad():
    real_index = np.where(real)[0]
    real_pred = torch.cat([synth_net(x_all[real_index[start:start + 256]]).argmax(dim=1)
                           for start in range(0, len(real_index), 256)]).numpy()

print(pd.crosstab(pd.Series(np.array(KIND_NAMES)[[KINDS.index(k) for k in real_kind[real]]], name="statement of PTB-XL"),
                  pd.Series(np.array(KIND_NAMES)[real_pred], name="network trained on synthetic ECGs")))

**Python note.** `has_code("AFIB")` marks the recordings whose statements hold AFIB. `np.select(conditions, choices, "other")` takes, for each recording, the choice of the first condition that holds, so a recording with SVT and AF counts as SVT. Normal ECGs need both NORM and SR, a normal ECG in sinus rhythm.

**Observe and reflect.** On its synthetic test recordings the network makes almost no error. On the real ECGs of PTB-XL it fails: Look at the row of the normal ECGs. Which differences between the synthetic and the real signals could explain this, and what would the network need before it could be used on real patients?

## 4. Label by label: AUC and thresholds

*Lecture: [Multi-label classification](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#many-labels-at-once-multi-label-classification)*

Every label of a multi-label network answers its own question, so it is judged on its own: an AUC on the test fold, and a threshold that turns its probability into a decision. The threshold is chosen on fold 9, where the sum of sensitivity and specificity is highest, and only then applied to fold 10.

**From the formulas to the code.** $p_k = \sigma(z_k)$: `torch.sigmoid(net(x))` inside `predict`. The threshold of label $k$ maximises $\mathrm{Se} + \mathrm{Sp} - 1$, the index of Youden, over the thresholds of the ROC curve: `thresholds[np.argmax(tpr - fpr)]` in `youden_threshold`. $\mathrm{PPV} = TP/(TP + FP)$: the third value of `se_sp_ppv`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 4 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB05_time_series_and_ecg_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(4, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
rows = []
fig, ax = plt.subplots(figsize=(4.6, 4.2))
for k, label in enumerate(RHYTHMS):
    threshold = youden_threshold(y_val[:, k], val_prob[:, k])
    se, sp, ppv = se_sp_ppv(y_test[:, k], test_prob[:, k] >= threshold)
    auc = roc_auc_score(y_test[:, k], test_prob[:, k])
    fpr, tpr, _ = roc_curve(y_test[:, k], test_prob[:, k])
    ax.plot(fpr, tpr, lw=1.2, label=f"{label} {auc:.3f}")
    rows.append({"label": label, "positives": int(y_test[:, k].sum()), "AUC": round(auc, 3),
                 "threshold": round(threshold, 3), "sensitivity": round(se, 3), "specificity": round(sp, 3), "PPV": round(ppv, 3)})
ax.plot([0, 1], [0, 1], ls="--", color="#999999", lw=0.8)
ax.set_xlabel("1 − specificity")
ax.set_ylabel("sensitivity")
ax.legend(fontsize=8)
plt.show()
label_table = pd.DataFrame(rows)
print(label_table)
print("Mean AUC on the test fold:", round(label_table["AUC"].mean(), 3))

**Python note.** `roc_curve` returns the false positive rate, the true positive rate and the thresholds at which they change; `test_prob[:, k] >= threshold` turns the probabilities of label `k` into decisions. `rows.append({...})` collects one dictionary per label, and `pd.DataFrame(rows)` makes a table of them.

In [ ]:
# What does the network say about the case?
case_prob = predict(rhythm_net, x_all[case:case + 1])[0]
print(pd.Series(case_prob.round(3), index=RHYTHMS))
print("Statements of the cardiologists:", ecgs.loc[case, "codes"])

**Observe and reflect.** The thresholds of the rare labels lie far below 0.5, and their positive predictive values stay low. For which label would you accept many false alarms, and for which would you rather miss some recordings? Think of what happens after an alarm in your own clinic.

**Exercise D.** What is the positive predictive value of the label STACH on the test fold when its probability is cut at 0.5 instead of its threshold from fold 9? Round to three decimals.

In [ ]:
data = exercise_table("D")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise D", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(se_sp_ppv(y_test[:, 2], test_prob[:, 2] >= 0.5)[2], 3)
teach.check("Exercise D", answer)

**Mini project: PhysioNet 2020 Challenge, nine rhythm disorders from the 12-lead ECG.** *Lecture: [Mini project: PhysioNet 2020 Challenge: Detection of nine rhythm disorders from the 12-lead ECG](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#mini-project-physionet-2020-challenge-detection-of-nine-rhythm-disorders-from-the-12-lead-ecg)*

The PhysioNet/Computing in Cardiology Challenge 2020 gathered 12-lead ECGs from six databases, and PTB-XL was one of them. Its first database, from the China Physiological Signal Challenge 2018 (CPSC 2018), labelled each recording with nine classes: normal, AF, first-degree atrioventricular block (I-AVB), left and right bundle branch block (LBBB, RBBB), premature atrial and ventricular contractions (PAC, PVC), ST depression (STD) and ST elevation (STE). The cells below find these nine classes among the statements of the PTB-XL part of the Challenge and train the network of Section 3 to recognise all nine at once.

**From the formulas to the code.** Nine outputs, each with its own probability $p_k = \sigma(z_k)$ and the binary cross-entropy: `RhythmNet(9)` and `nn.BCEWithLogitsLoss()`. `fit_labels` holds the training loop of Section 3 in a function, so that it can train any multi-label network on folds 1 and 2.

In [ ]:
CPSC = {"NORM": "Normal", "AFIB": "AF", "1AVB": "I-AVB", "CLBBB": "LBBB", "CRBBB": "RBBB",
        "PAC": "PAC", "PVC": "PVC", "STD_": "STD", "STE_": "STE"}
labels9 = ptbxl.labels(ecgs, list(CPSC))
y9_all = torch.tensor(np.array(labels9, dtype=np.float32))
y9_val, y9_test = labels9.to_numpy()[va], labels9.to_numpy()[te]


def fit_labels(net, y_train, y_val, epochs=15, batch=64):
    """Trains a multi-label network on folds 1 and 2 and keeps the epoch with the best mean AUC on fold 9."""
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.003, total_steps=epochs * math.ceil(len(x_train) / batch))
    loss_fn = nn.BCEWithLogitsLoss()
    best_auc, best_epoch, best_state = -1.0, 0, None
    for epoch in range(1, epochs + 1):
        net.train()
        order = torch.randperm(len(x_train))
        for i in range(0, len(order), batch):
            b = order[i:i + batch]
            optimizer.zero_grad()
            loss = loss_fn(net(x_train[b]), y_train[b])
            loss.backward()
            optimizer.step()
            scheduler.step()
        val_auc = macro_auc(y_val, predict(net, x_val))
        if val_auc > best_auc:
            best_auc, best_epoch, best_state = val_auc, epoch, copy.deepcopy(net.state_dict())
    net.load_state_dict(best_state)
    return best_epoch, best_auc

torch.manual_seed(RANDOM_STATE)
net9 = RhythmNet(len(CPSC))
best_epoch9, best_auc9 = fit_labels(net9, y9_all[tr], y9_val)
test_prob9 = predict(net9, x_test)
table9 = pd.DataFrame({"fold 1-2": labels9.to_numpy()[tr].sum(axis=0),
                       "fold 10": y9_test.sum(axis=0).astype(int),
                       "AUC": [round(roc_auc_score(y9_test[:, k], test_prob9[:, k]), 3) for k in range(len(CPSC))]},
                      index=list(CPSC.values()))

print("Best epoch:", best_epoch9, "· mean AUC on fold 9:", round(best_auc9, 3))
print(table9)
print("Mean AUC of the nine classes on fold 10:", round(table9["AUC"].mean(), 3))

**Python note.** `CPSC` is a dictionary from the statements of PTB-XL to the names of the nine classes; `list(CPSC)` gives its keys and `CPSC.values()` its names. The function `fit_labels` keeps its variables inside, so the loop does not overwrite the optimizer or the loss of Section 3. `table9` puts the number of recordings of each class in the training folds and in fold 10 next to its AUC.

**Observe and reflect.** Which classes does the network recognise almost perfectly, and which hardly better than chance? Compare the AUC of each class with the number of its recordings in the training folds. What would you do first for the weakest class: more recordings, another threshold, or another input?

## 5. Recurrent networks on the RR intervals: GRU and LSTM

*Lecture: [LSTM: Capturing temporal dependencies](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#recurrent-networks-a-state-that-moves-along-the-signal)*

A recurrent network reads a sequence one element at a time and carries a state from step to step. Here the sequence is the RR intervals of a recording, between 7 and 32 numbers, and the network is a gated recurrent unit (GRU) with a state of 16 numbers. It sees nothing of the waves of the ECG, only the timing of its beats.

**From the formulas to the code.** $h_t = (1 - z_t)\, h_{t-1} + z_t\, \tilde{h}_t$, with the update gate $z_t$: `nn.GRU(1, 16, batch_first=True)`. Recordings have different numbers of intervals, so `rr_sequences` fills each sequence with zeros up to 32 steps and keeps its true length, and `pack_padded_sequence` tells the GRU to stop at the true length.

**Started here?** Run the catch-up cell below once. It runs the code that Section 5 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB05_time_series_and_ecg_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(5, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
MAX_INTERVALS = 32


def rr_sequences(rr_list):
    """RR intervals as sequences of equal length for the GRU, scaled as (RR - 0.8) / 0.2, and the true length of each."""
    seq = np.zeros((len(rr_list), MAX_INTERVALS, 1), dtype=np.float32)
    lengths = np.ones(len(rr_list), dtype=np.int64)
    for i, rr in enumerate(rr_list):
        rr = np.asarray(rr[:MAX_INTERVALS])
        seq[i, :len(rr), 0] = (rr - 0.8) / 0.2
        lengths[i] = max(1, len(rr))
    return torch.tensor(seq), torch.tensor(lengths)


class RRNet(nn.Module):
    """A GRU of 16 numbers that reads the RR intervals one by one, and one output per label from its last state."""

    def __init__(self, hidden=16, n_labels=6):
        super().__init__()
        self.gru = nn.GRU(1, hidden, batch_first=True)
        self.out = nn.Linear(hidden, n_labels)

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        _, h = self.gru(packed)
        return self.out(h[-1])

seq_all, len_all = rr_sequences(rr_list)
torch.manual_seed(RANDOM_STATE)
rr_net = RRNet(n_labels=len(RHYTHMS))
optimizer = torch.optim.Adam(rr_net.parameters(), lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()
train_index = np.where(tr)[0]
best_auc_rr, best_epoch_rr = -1.0, 0
for epoch in range(1, 61):
    rr_net.train()
    order = train_index[torch.randperm(len(train_index)).numpy()]
    for i in range(0, len(order), 64):
        b = order[i:i + 64]
        optimizer.zero_grad()
        loss = loss_fn(rr_net(seq_all[b], len_all[b]), y_all[b])
        loss.backward()
        optimizer.step()
    val_auc = macro_auc(y_val, predict(rr_net, seq_all[va], len_all[va]))
    if val_auc > best_auc_rr:
        best_auc_rr, best_epoch_rr, best_state_rr = val_auc, epoch, copy.deepcopy(rr_net.state_dict())
rr_net.load_state_dict(best_state_rr)
test_prob_rr = predict(rr_net, seq_all[te], len_all[te])

print("Parameters of the GRU network:", sum(p.numel() for p in rr_net.parameters()), "· best epoch", best_epoch_rr)
comparison = pd.DataFrame({"rule (CV)": [np.nan, round(roc_auc_score(af[te & has_cv], cv[te & has_cv]), 3), np.nan, np.nan, np.nan, np.nan],
                           "convolutional network": label_table["AUC"].to_numpy(),
                           "GRU": [round(roc_auc_score(y_test[:, k], test_prob_rr[:, k]), 3) for k in range(len(RHYTHMS))]}, index=RHYTHMS)
print(comparison)
print("Mean AUC:", comparison.iloc[:, 1:].mean().round(3).to_dict())

**Python note.** `h[-1]` is the last state of the GRU, after the true last interval of each recording, and `self.out` turns its 16 numbers into six scores. `(rr - 0.8) / 0.2` scales the intervals around a typical value, which makes the training faster. 60 epochs over 4,356 short sequences take about half a minute. `predict(rr_net, seq_all[va], len_all[va])` passes two inputs to the network, the sequences and their lengths.

**Observe and reflect.** With about a thousand weights and without seeing a single wave, the GRU matches or beats the convolutional network on most labels. On which label does the convolutional network keep its lead, and which part of the ECG explains it?

**Exercise E.** How many parameters does the recurrent network have when its state holds 32 numbers instead of 16, `RRNet(hidden=32)`?

In [ ]:
answer = None   # your answer
teach.check("Exercise E", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = sum(p.numel() for p in RRNet(hidden=32).parameters())
teach.check("Exercise E", answer)

**LSTM: a second memory for long dependencies.** *Lecture: [LSTM: Capturing temporal dependencies](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#recurrent-networks-a-state-that-moves-along-the-signal)*

The long short-term memory (LSTM) carries two states from step to step: the hidden state $h_t$ and a cell state $c_t$, which forget, input and output gates protect. The cells below train an LSTM of the same size on the same RR intervals and compare it with the GRU.

**From the formulas to the code.** $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$ and $h_t = o_t \odot \tanh(c_t)$: `nn.LSTM(1, 16, batch_first=True)`, which returns the last $h$ and the last $c$ of every sequence as `(h, c)`.

In [ ]:
class RRLSTM(nn.Module):
    """An LSTM of 16 numbers that reads the RR intervals one by one, and one output per label from its last state."""

    def __init__(self, hidden=16, n_labels=6):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, batch_first=True)
        self.out = nn.Linear(hidden, n_labels)

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        _, (h, c) = self.lstm(packed)
        return self.out(h[-1])


def fit_rr(net, epochs=60):
    """Trains a network on the RR sequences of folds 1 and 2 and keeps the epoch with the best mean AUC on fold 9."""
    optimizer = torch.optim.Adam(net.parameters(), lr=0.01)
    loss_fn = nn.BCEWithLogitsLoss()
    train_index = np.where(tr)[0]
    best_auc, best_epoch, best_state = -1.0, 0, None
    for epoch in range(1, epochs + 1):
        net.train()
        order = train_index[torch.randperm(len(train_index)).numpy()]
        for i in range(0, len(order), 64):
            b = order[i:i + 64]
            optimizer.zero_grad()
            loss = loss_fn(net(seq_all[b], len_all[b]), y_all[b])
            loss.backward()
            optimizer.step()
        val_auc = macro_auc(y_val, predict(net, seq_all[va], len_all[va]))
        if val_auc > best_auc:
            best_auc, best_epoch, best_state = val_auc, epoch, copy.deepcopy(net.state_dict())
    net.load_state_dict(best_state)
    return best_epoch, best_auc


torch.manual_seed(RANDOM_STATE)
lstm_net = RRLSTM(n_labels=len(RHYTHMS))
best_epoch_lstm, best_auc_lstm = fit_rr(lstm_net)
test_prob_lstm = predict(lstm_net, seq_all[te], len_all[te])

print("Parameters (GRU, LSTM):", sum(p.numel() for p in rr_net.parameters()), sum(p.numel() for p in lstm_net.parameters()),
      "· best epoch of the LSTM", best_epoch_lstm)
comparison["LSTM"] = [round(roc_auc_score(y_test[:, k], test_prob_lstm[:, k]), 3) for k in range(len(RHYTHMS))]
print(comparison)
print("Mean AUC:", comparison.iloc[:, 1:].mean().round(3).to_dict())

**Python note.** `fit_rr` holds the training loop of the GRU in a function, so that any network that reads the RR sequences can be trained the same way. `_, (h, c) = self.lstm(packed)` unpacks the two states that the LSTM returns; only the hidden state of the last layer, `h[-1]`, goes to the output layer. `comparison["LSTM"] = [...]` adds a column to the table of the GRU.

**Observe and reflect.** The LSTM has a cell state and three gates, the GRU two gates and no cell state. On which labels does the extra memory help, and is the difference larger than what a second training with another seed might give?

**The Python path of Practice stop 1 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#last-two-hours-advanced-ecg-models-and-clinical-integration) and continue reading at «Last two hours: Advanced ECG models and clinical integration».

## 6. The median beat and the J point

*Lecture: [Measuring the ST segment automatically: The median beat and the J point](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#measuring-the-st-segment-automatically-the-median-beat-and-the-j-point)*

Before it measures, the software of an ECG machine builds one clean beat for each lead. It aligns the single beats on their R peaks and takes their median sample by sample, so that noise and a single premature beat lose their weight. The QRS onset and the J point come from the slopes of all twelve leads together, and the reference level of each lead is the PR segment just before the QRS onset.

**From the formulas to the code.** $\tilde{x}_\ell(\tau) = \mathrm{median}_i\, x_\ell(t_i + \tau)$ over the R peaks $t_i$: `np.median(single, axis=0)`. $\mathrm{ST}_\ell = \tilde{x}_\ell(\tau_J) - \mathrm{PR}_\ell$: `case_beat[:, j_point] - level`, where `level` is the mean of the 40 ms that end 20 ms before the QRS onset. `beats.qrs_onset_offset` finds the onset and the J point from the global slope, the sum of the absolute slopes of the twelve leads.

**Started here?** Run the catch-up cell below once. It runs the code that Section 6 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB05_time_series_and_ecg_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(6, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
# The median beat of the case, step by step
before, after = 30, 56                   # 0.30 s before and 0.55 s after the R peak, at 100 Hz
single = np.stack([signals[case][:, p - before:p + after] for p in r_peak_list[case] if p - before >= 0 and p + after <= 1000])
case_beat = np.median(single, axis=0)
onset, j_point = beats.qrs_onset_offset(case_beat)
level = case_beat[:, onset - 6:onset - 2].mean(axis=1)
st_case = case_beat[:, j_point] - level
print(single.shape, "→", case_beat.shape)
print("QRS onset and J point, ms from the R peak:", (onset - before) * 10, (j_point - before) * 10)
print(pd.Series(st_case.round(3), index=beats.LEADS))

**Python note.** `single` has the shape (beats, leads, samples), and `np.median(single, axis=0)` takes the median over the first axis, the beats, separately for every lead and sample. `case_beat[:, onset - 6:onset - 2]` takes the four samples from 60 to 30 ms before the onset, the 40 ms of the PR segment, and `.mean(axis=1)` averages them for each lead. Subtracting a vector of 12 levels from the 12 values at the J point works lead by lead.

In [ ]:
ms = np.arange(-before, after) * 10
v2 = beats.LEADS.index("V2")
fig, ax = plt.subplots(figsize=(6, 3.4))
for s in single:
    ax.plot(ms, s[v2], color="#C8C8C8", lw=0.7)
ax.plot(ms, case_beat[v2], color="#231A1A", lw=1.8, label="median beat of V2")
ax.axvspan((onset - 6 - before) * 10, (onset - 3 - before) * 10, color="#B97813", alpha=0.2, label="PR segment, 40 ms")
ax.axvline((j_point - before) * 10, color="#9B1B1B", ls="--", label="J point")
ax.axhline(level[v2], color="#B97813", lw=0.8)
ax.set_xlabel("time from the R peak (ms)")
ax.set_ylabel("mV")
ax.legend(fontsize=8)
plt.show()
alarm, meet, groups = beats.stemi_criteria(dict(zip(beats.LEADS, st_case)), "M", 68)
print("Alarm:", alarm, "· leads:", meet, "· groups:", groups)

**Observe and reflect.** The QRS complex of the case lasts 120 ms, and the J point lies 50 ms after the R peak. How would the ST deviation of V2 change if the program placed the J point 20 ms earlier, inside the downstroke of the S wave? Which kind of error, too early or too late, would cause false alarms?

**Exercise F.** Some algorithms measure the ST segment 40 ms after the J point. What is the ST deviation of V3 of the case there, in mV? Round to three decimals.

In [ ]:
data = exercise_table("F")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise F", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
v3 = beats.LEADS.index("V3")
answer = round(float(case_beat[v3, j_point + 4] - level[v3]), 3)
teach.check("Exercise F", answer)

## 7. The ST deviation and the STEMI alarm on folds 9 and 10

*Lecture: [ST elevation and depression: The criteria of the universal definition](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#st-elevation-and-depression-the-criteria-of-the-universal-definition)*

The alarm of the course applies the criteria of the fourth universal definition to the median beat of every recording: new ST elevation at the J point of at least 1 mm in two contiguous leads, with the cut-points of V2 and V3 by sex and age. The truth is the stage of the infarction given by the cardiologists of PTB-XL: Stage I and stage I-II count as acute.

**From the formulas to the code.** The cut-point of lead $\ell$ is `beats.cut_point(lead, sex, age)`, 0.2, 0.25 or 0.15 mV in V2 and V3 and 0.1 mV elsewhere. A group fires when at least two of its leads reach their cut-points: `sum(n in meet for n in names) >= 2` in `beats.stemi_criteria`. Sensitivity, specificity and PPV come from `se_sp_ppv(st_table["acute"], st_table["alarm"])`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 7 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB05_time_series_and_ecg_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(7, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
st_rows, median_beats = [], {}
for i in np.where(va | te)[0]:
    beat, n_beats = beats.median_beat(signals[i], peaks=r_peak_list[i])
    median_beats[i] = beat
    st = beats.st_deviation(beat)
    alarm, meet, groups = beats.stemi_criteria(st["st_mv"], ecgs["sex"].iloc[i], ecgs["age"].iloc[i])
    st_rows.append({"ecg_id": int(ecgs["ecg_id"].iloc[i]), "split": split[i], "sex": ecgs["sex"].iloc[i], "age": ecgs["age"].iloc[i],
                    "acute": bool(ecgs["acute_mi"].iloc[i]), **st["st_mv"], "alarm": alarm, "groups": " ".join(groups)})
st_table = pd.DataFrame(st_rows)
for name, part in (("folds 9 and 10", st_table), ("test", st_table[st_table["split"] == "test"])):
    se, sp, ppv = se_sp_ppv(part["acute"], part["alarm"])
    print(f"{name}: {len(part)} recordings · {int(part['acute'].sum())} acute infarctions · {int(part['alarm'].sum())} alarms"
          f" · sensitivity {se:.3f} · specificity {sp:.3f} · PPV {ppv:.3f}")

**Python note.** The loop builds the median beat of every recording of folds 9 and 10 once, about half a minute, and keeps it in the dictionary `median_beats` for the QT of Section 8. `**st["st_mv"]` unpacks the dictionary of the twelve ST deviations into the row, so that each lead becomes a column of `st_table`.

In [ ]:
codes_of = ecgs.set_index("ecg_id")["codes"]
false_alarm = st_table["alarm"] & ~st_table["acute"]
counts = Counter(c for e in st_table.loc[false_alarm, "ecg_id"] for c in codes_of[e])
print("False alarms:", int(false_alarm.sum()))
print(pd.Series(dict(counts.most_common(12))))
for code_name in ("CLBBB", "PACE", "LVH", "NORM"):
    has_code = st_table["ecg_id"].map(lambda e, c=code_name: c in codes_of[e])
    print(f"{code_name}: {int(st_table.loc[has_code, 'alarm'].sum())} alarms in", int(has_code.sum()), "recordings")
missed = st_table[st_table["acute"] & ~st_table["alarm"]]
print("Missed acute infarctions:", [(e, sorted(codes_of[e])) for e in missed["ecg_id"]])

**Observe and reflect.** Nearly every recording with complete LBBB raises the alarm, and so do most paced recordings. The universal definition says that its ST criteria do not apply to them. What should an automatic alarm do with such a recording: stay silent, ask for a human reading, or apply other criteria?

In [ ]:
def alarm_with_margin(row, margin):
    """The alarm of the universal definition with every cut-point raised by margin (mV)."""
    meet = [lead for lead in beats.LEADS if row[lead] >= beats.cut_point(lead, row["sex"], row["age"]) + margin - 1e-9]
    return any(sum(n in meet for n in names) >= 2 for names in beats.GROUPS.values())


for margin in (0.0, 0.05, 0.10):
    alarm_m = st_table.apply(alarm_with_margin, axis=1, margin=margin)
    se, sp, ppv = se_sp_ppv(st_table["acute"], alarm_m)
    print(f"+{margin:.2f} mV: {int(alarm_m.sum())} alarms · sensitivity {se:.3f} · PPV {ppv:.3f}")

**Exercise G.** How many alarms does the alarm give on folds 9 and 10 when every cut-point is raised by 0.05 mV?

In [ ]:
data = exercise_table("G")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise G", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int(st_table.apply(alarm_with_margin, axis=1, margin=0.05).sum())
teach.check("Exercise G", answer)

**The Python path of Practice stop 2 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#qt-prolongation-and-its-risk) and continue reading at «QTc prolongation risk algorithm».

## 8. The QT interval on the median beat and a QTc risk algorithm

*Lecture: [QTc prolongation risk algorithm](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#qt-prolongation-and-its-risk)*

The QT interval runs from the QRS onset to the end of the T wave. The course takes the onset from all leads and the end of the T wave from V5 by the tangent method: The tangent at the steepest point of the descending limb of the T wave meets the baseline at the end of the wave. The QT interval is then corrected for the heart rate.

**From the formulas to the code.** $\mathrm{QTc}_B = \mathrm{QT}/\sqrt{RR}$ and $\mathrm{QTc}_F = \mathrm{QT}/\sqrt[3]{RR}$, with RR in seconds: `beats.qtc(qt, rr, "Bazett")` and `beats.qtc(qt, rr, "Fridericia")`. `beats.qt_interval(beat, rr_s=...)` measures the QT interval in ms on a median beat; it gives *not a number* when no end of the T wave is found.

**Started here?** Run the catch-up cell below once. It runs the code that Section 8 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB05_time_series_and_ecg_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(8, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
qt_rows = []
for i, beat in median_beats.items():
    rr_mean = float(rr_list[i].mean()) if len(rr_list[i]) else np.nan
    qt_rows.append({"ecg_id": int(ecgs["ecg_id"].iloc[i]), "qt_ms": beats.qt_interval(beat, rr_s=rr_mean), "rr_s": rr_mean,
                    "long_qt": "LNGQT" in ecgs["codes"].iloc[i]})
qt_table = pd.DataFrame(qt_rows)
qt_table["qtc_b"] = beats.qtc(qt_table["qt_ms"], qt_table["rr_s"], "Bazett")
qt_table["qtc_f"] = beats.qtc(qt_table["qt_ms"], qt_table["rr_s"], "Fridericia")
print("Recordings without a QT interval:", int(qt_table["qt_ms"].isna().sum()))
print(qt_table.groupby("long_qt")[["qtc_b", "qtc_f"]].median().round(0))
print("QTc of 500 ms or more (Bazett, Fridericia):", int((qt_table["qtc_b"] >= 500).sum()), int((qt_table["qtc_f"] >= 500).sum()))
fig, ax = plt.subplots(figsize=(6, 3))
bins = np.arange(300, 650, 10)
ax.hist(qt_table.loc[~qt_table["long_qt"], "qtc_b"].dropna(), bins=bins, color="#1F5F8B", alpha=0.6, label="other recordings")
ax.hist(qt_table.loc[qt_table["long_qt"], "qtc_b"].dropna(), bins=bins, color="#9B1B1B", alpha=0.8, label="statement of a long QT")
ax.axvline(500, color="k", ls="--", lw=1)
ax.set_yscale("log")
ax.set_xlabel("QTc by Bazett (ms)")
ax.legend(fontsize=8)
plt.show()

**Python note.** `median_beats.items()` gives the pairs of a recording and its median beat that Section 7 kept, so no beat is built twice. `beats.qtc(qt_table["qt_ms"], qt_table["rr_s"], ...)` works on whole columns at once. A comparison with *not a number*, such as `qtc_b >= 500`, gives `False`, so a missing QT interval never counts as long.

**Observe and reflect.** Most recordings with a QTc of 500 ms or more by Bazett carry no statement of a long QT interval. Which of them would you check first by hand: those with a fast heart rate, those with a wide QRS complex, or those with a flat T wave in V5? Why?

**Exercise H.** How many recordings of folds 9 and 10 have a QTc by Fridericia of 500 ms or more?

In [ ]:
data = exercise_table("H")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise H", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int((qt_table["qtc_f"] >= 500).sum())
teach.check("Exercise H", answer)

**A QTc prolongation risk algorithm.** *Lecture: [QTc prolongation risk algorithm](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#qt-prolongation-and-its-risk)*

An automatic QTc becomes more useful when it enters a risk algorithm together with the clinical picture. The risk score of Tisdale and colleagues predicts a QTc prolongation during a stay in a cardiac care unit from ten variables that every ward knows, one of them the QTc at admission. The cells below compute it for four inpatients of the virtual clinic: their age, sex and diagnosis come from the registry, their QTc from the median beat of their synthetic ECG, and their medication, potassium and sepsis from a ward scenario of the course.

**From the formulas to the code.** The score adds points: 1 each for an age of 68 years or more, female sex and a loop diuretic; 2 each for a potassium of 3.5 mmol/L or less, an admission QTc of 450 ms or more and an acute myocardial infarction; 3 each for one drug that prolongs the QT interval, a second such drug, sepsis and heart failure. `tisdale_score(...)` returns the sum, 0 to 21, and the risk group: low up to 6, moderate from 7 to 10 and high from 11.

In [ ]:
def tisdale_score(age, female, loop_diuretic, potassium, qtc_ms, acute_mi, qt_drugs, sepsis, heart_failure):
    """Points of the risk score of Tisdale and colleagues for QTc prolongation in hospital, and the risk group."""
    points = (1 * (age >= 68) + 1 * female + 1 * loop_diuretic + 2 * (potassium <= 3.5) + 2 * (qtc_ms >= 450)
              + 2 * acute_mi + 3 * (qt_drugs >= 1) + 3 * (qt_drugs >= 2) + 3 * sepsis + 3 * heart_failure)
    group = "low" if points <= 6 else ("moderate" if points <= 10 else "high")
    return int(points), group


def automatic_qtc(ecg_array):
    """The QTc by Bazett, in ms, measured on the median beat of a 12-lead ECG at 100 Hz."""
    peaks = beats.r_peaks_many(ecg_array[None])[0]
    rr_s = float(np.mean(np.diff(peaks))) / 100
    beat, _ = beats.median_beat(ecg_array, peaks=peaks)
    return float(beats.qtc(beats.qt_interval(beat, rr_s=rr_s), rr_s, "Bazett"))

ward_cohort = vclinic.make_cohort(500)
ward = pd.DataFrame({"patient_id": ["VC-0335", "VC-0009", "VC-0183", "VC-0002"],
                     "loop_diuretic": [1, 1, 0, 0], "potassium": [3.4, 3.9, 4.1, 4.0],
                     "qt_drugs": [1, 2, 0, 1], "sepsis": [0, 0, 0, 1]})
ward = ward.merge(ward_cohort[["patient_id", "age", "sex", "main_diagnosis"]], on="patient_id")
ward["qtc_auto"] = [round(automatic_qtc(vclinic.patient_ecg(ward_cohort, p, fs=100).leads.to_numpy().T.astype(np.float32)))
                    for p in ward["patient_id"]]
scores = [tisdale_score(r.age, r.sex == "F", r.loop_diuretic, r.potassium, r.qtc_auto, r.main_diagnosis == "ACS",
                        r.qt_drugs, r.sepsis, r.main_diagnosis in ("HFrEF", "HFpEF")) for r in ward.itertuples()]
ward["points"], ward["risk"] = [p for p, _ in scores], [g for _, g in scores]
print(ward.assign(sex=ward["sex"].map({"M": "M", "F": "F"}), main_diagnosis=ward["main_diagnosis"].replace("none", "none")).rename(columns={"patient_id": "patient", "loop_diuretic": "loop diuretic", "potassium": "potassium",
                           "qt_drugs": "QT drugs", "age": "age", "sex": "sex", "main_diagnosis": "diagnosis",
                           "qtc_auto": "QTc (ms)", "points": "points"}).to_string(index=False))

**Python note.** `True` counts as 1 and `False` as 0 in a sum, so `1 * (age >= 68)` adds one point only when the condition holds. `ward.merge(..., on="patient_id")` adds the age, the sex and the diagnosis of the registry to the rows of the scenario. `ward.itertuples()` gives one row at a time with its columns as names, such as `r.age`. `automatic_qtc` repeats the measurement of this section on one ECG: R peaks, median beat, QT interval and the correction of Bazett.

**Observe and reflect.** Patient VC-0183 has the longest QTc of the ward and the lowest score. Why does this not contradict the score, and what should happen with her QTc anyway? Patient VC-0002 has an automatic QTc just below 450 ms: How many points would an error of a few milliseconds change, and what does this mean for a threshold built into an algorithm?

## 9. Long-term monitoring: a day of the virtual clinic

*Lecture: [Holter analysis: Long-term AI monitoring](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#long-term-monitoring-from-ten-seconds-to-twenty-four-hours)*

A Holter recorder of the virtual clinic has followed patient VC-0128 for 24 hours, from 08:00. The recording is synthetic: three episodes of paroxysmal AF, an hour of frequent atrial premature beats, two periods of electrode noise and the slow rise and fall of the rate over the day. Its detector of beats misses some beats and finds others that were not there, as real detectors do in noise. Because the truth is known beat by beat, every alarm can be checked.

**From the formulas to the code.** $\text{AF burden} = \frac{n_{AF} \times L}{24 \times 3600}$, with $n_{AF}$ windows with an alarm of length $L$ seconds: `100 * alarms * length / (24 * 3600)`. The window of a beat is `time // length`, the whole part of the division, and `groupby` collects the RR intervals of each window.

**Started here?** Run the catch-up cell below once. It runs the code that Section 9 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB05_time_series_and_ecg_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(9, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
holter_rec = holter.recording()
b = holter_rec["detected_s"]
truth_af = np.zeros(len(b), dtype=int)
for s, e in holter_rec["episodes"]:
    truth_af[(b >= s) & (b <= e)] = 1
holter_beats = pd.DataFrame({"time": b[1:], "RR": np.diff(b), "AF": truth_af[1:]})
print(len(holter_beats), "beats · true AF burden:", round(100 * holter.burden(holter_rec), 1), "%")
for s, e in holter_rec["episodes"]:
    print("AF:", holter.clock_text(s), "–", holter.clock_text(e))
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.plot(holter_beats["time"] / 3600, holter_beats["RR"], ",", color="#1F5F8B")
for s, e in holter_rec["episodes"]:
    ax.axvspan(s / 3600, e / 3600, color="#B97813", alpha=0.25)
ax.set_xticks(range(0, 25, 4))
ax.set_xticklabels([f"{(8 + h) % 24:02d}:00" for h in range(0, 25, 4)])
ax.set_ylim(0, 2)
ax.set_ylabel("RR (s)")
plt.show()

**Python note.** `holter.recording()` returns a dictionary; `holter_rec["detected_s"]` holds the times of the beats as the detector found them, and `holter_rec["episodes"]` the start and the end of every episode of AF in seconds. `(b >= s) & (b <= e)` marks the beats inside one episode. The plot shows more than 100,000 intervals, one point each (`","` draws a pixel).

In [ ]:
# Four strips of lead II from the day
fig, axes = plt.subplots(4, 1, figsize=(10, 6.4), sharex=True)
for ax, (clock, title) in zip(axes, [("08:30", "sinus rhythm"), ("13:25", "AF"),
                                     ("18:30", "atrial premature beats"), ("07:05", "electrode noise")]):
    start = ((int(clock[:2]) + int(clock[3:]) / 60 - 8) % 24) * 3600
    t, strip = holter.strip(holter_rec, start, 10)
    teach.plot_strip(ax, t, strip, label=f"{clock} · {title}", v_range=(-1.5, 2.5))
plt.show()

In [ ]:
def window_table(beats_df, length):
    """Windows of length seconds: the coefficient of variation of the RR intervals, the truth and the rule for quality."""
    w = (beats_df["time"] // length).astype(int)
    g = beats_df.groupby(w)
    out = pd.DataFrame({"cv": g["RR"].std() / g["RR"].mean(), "af": g["AF"].mean() > 0.5,
                        "bad": g["RR"].apply(lambda r: bool(((r < 0.3) | (r > 1.6)).any()))})
    out["hour"] = ((8 + out.index * length / 3600) % 24).astype(int)
    return out


rows = []
for length in (30, 60, 120):
    wt = window_table(holter_beats, length)
    for quality in (False, True):
        use = ~wt["bad"] if quality else pd.Series(True, index=wt.index)
        alarm = (wt["cv"] > 0.10) & use
        rows.append({"window (s)": length, "quality rule": quality, "alarms": int(alarm.sum()),
                     "false": int((alarm & ~wt["af"]).sum()), "missed": int((~alarm & wt["af"]).sum()),
                     "burden (%)": round(100 * alarm.sum() * length / (24 * 3600), 1)})
print(pd.DataFrame(rows))
wt30 = window_table(holter_beats, 30)
false_30 = (wt30["cv"] > 0.10) & ~wt30["af"]
print("False alarms of windows of 30 s by hour:", wt30[false_30].groupby("hour").size().to_dict())

**Python note.** `window_table` gives one row per window. `g["RR"].apply(lambda r: ...)` runs a small function on the intervals of every window: `lambda r:` defines a function without a name, here one that asks whether any interval is shorter than 0.3 s or longer than 1.6 s. `use` keeps all windows, or only those that are not bad when the quality rule is on; a window left out cannot raise an alarm, so a window of AF that the rule leaves out counts as missed. The true burden is 15.4 percent; compare each row with it.

**Observe and reflect.** With windows of 120 seconds and the quality rule, no false alarm remains, but the estimated burden falls below the truth. Which windows of AF does the rule leave out, and why? What would you change so that it keeps them?

**Exercise I.** What is the AF burden of the day with windows of 60 seconds and the quality rule, in percent with one decimal?

In [ ]:
data = exercise_table("I")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise I", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
wt60 = window_table(holter_beats, 60)
answer = round(100 * int(((wt60["cv"] > 0.10) & ~wt60["bad"]).sum()) * 60 / (24 * 3600), 1)
teach.check("Exercise I", answer)

**The Python path of Practice stop 3 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#ecg-algorithms-with-regulatory-approval) and continue reading at «Overview of FDA/CE-approved ECG AI products». The rest of this Jupyter notebook holds further practice and the help for the weekly project.

## 10. Working with an AI assistant, part 5: Reviewing a time-series pipeline

*Lecture: [Working with an AI assistant, part 5: Reviewing a time-series pipeline](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lecture.html#ai-pair-programming-part-5-reviewing-a-time-series-pipeline)*

A participant asked an AI assistant to *train a detector of AF from the RR intervals of lead II of PTB-XL and to report how good it is*. The code below is its answer. It runs and prints an accuracy that looks good. Before trusting it, read it with the six questions of the review of a time-series pipeline: Are the recordings of one patient kept in one set? Are the sampling frequency and the units right? Are the leads taken by their names? Were the thresholds chosen on the validation fold? Are the results reported for each label, with its prevalence? Is the rule of the measurement stated?

**Started here?** Run the catch-up cell below once. It runs the code that Section 10 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB05_time_series_and_ecg_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-05/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(10, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
# The answer of the AI assistant
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

lead_ii_all = signals[:, 0]                          # lead II
peaks_ai = [beats.r_peaks(s, fs=500) for s in lead_ii_all]
rr_ai = [np.diff(p) / 500 for p in peaks_ai]
features_ai = np.array([[np.mean(r), np.std(r)] if len(r) > 1 else [0.0, 0.0] for r in rr_ai])
y_ai = labels["AFIB"].to_numpy()
f_train, f_test, y_train_ai, y_test_ai = train_test_split(features_ai, y_ai, test_size=0.3, random_state=0)
model_ai = LogisticRegression().fit(f_train, y_train_ai)
print("Accuracy:", round(model_ai.score(f_test, y_test_ai), 3))
prob_ai = model_ai.predict_proba(f_test)[:, 1]
fpr_ai, tpr_ai, thr_ai = roc_curve(y_test_ai, prob_ai)
k_ai = int(np.argmax(tpr_ai - fpr_ai))
print("Best threshold:", round(float(thr_ai[k_ai]), 3), "· sensitivity:", round(float(tpr_ai[k_ai]), 3))

The code has four problems and one omission. `signals[:, 0]` is lead I, not lead II: The leads of PTB-XL are stored in the order I, II, III, and a lead should be taken by its name, `beats.LEADS.index("II")`. `fs=500` tells the detector that the signal has 500 samples per second, while the recordings of the course have 100: Its filters and its refractory period work at the wrong scale, and every interval comes out five times too short. `train_test_split` mixes the recordings of one patient between training and test and ignores the folds of PTB-XL. The threshold is chosen on the test set itself. The omission: An accuracy hides how rare AF is. A program that never raises an alarm reaches an accuracy of 0.93 on these recordings, because 93 percent of them have no AF.

**Exercise J.** Correct the pipeline: Use the RR intervals of Section 2 from all twelve leads at 100 Hz and the same two features, the mean and the standard deviation with `np.std`. Train on folds 1 and 2 with standardised features, choose the threshold on fold 9 and report the sensitivity on fold 10, rounded to three decimals.

In [ ]:
data = exercise_table("J")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise J", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
features_rr = np.array([[np.mean(r), np.std(r)] if len(r) > 1 else [0.0, 0.0] for r in rr_list])
model_rr = make_pipeline(StandardScaler(), LogisticRegression()).fit(features_rr[tr], af[tr])
threshold_rr = youden_threshold(af[va], model_rr.predict_proba(features_rr[va])[:, 1])
answer = round(se_sp_ppv(af[te], model_rr.predict_proba(features_rr[te])[:, 1] >= threshold_rr)[0], 3)
teach.check("Exercise J", answer)

The [prompt workshop](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lab.html#part-f) of the interactive lab writes the prompt of such a pipeline with the seven fields of the prompt card. The field *Data* names the sampling frequency, the order of the leads and the folds, and the field *Test* asks for the sensitivity and the positive predictive value next to the accuracy.

## Application challenges by field

| Field | Challenge |
|---|---|
| Electrophysiology | The rule of Section 2 and the GRU of Section 5 both read only the RR intervals. Raise the threshold of the rule until its specificity on fold 9 equals that of the GRU for AF at its threshold, and compare the two sensitivities on fold 10. Which would you trust in a patient with frequent premature beats? |
| Interventional cardiology | Look at the median beats of the acute infarctions that the alarm of Section 7 missed. How many show ST depression in V1 to V3, and what would the posterior leads V7 to V9 add to the alarm? |
| Chest pain unit | With the positive predictive value of Section 7, how many of 50 alarms in a night would be true? Write down what a unit should do with an alarm before the catheterisation team is called. |
| Heart failure | Left bundle branch block is common in heart failure. Compute the share of the alarms of Section 7 that come from recordings with complete LBBB, and propose a rule for them, for example the criteria of Sgarbossa. |
| Intensive care | Many drugs used in intensive care prolong the QT interval. Use `qt_table` of Section 8: How many recordings reach 500 ms by Bazett but not by Fridericia, and at which heart rates? |
| Cardio-oncology | A patient has a QTc of 470 ms before a drug that prolongs the QT interval and 505 ms after its start. Given the error of the measurement of the course on the virtual clinic, a standard deviation of about 16 ms, how large a change would you call real? |
| Rhythm clinic | Change the quality rule of Section 9 so that it keeps the windows of fast AF and still removes the false alarms of the noisy periods. Which feature of the noise did you use, and what is the AF burden now? |

## After the notebook

Open the [interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-05/lab.html) and take the self-assessment in the tab *Check yourself*. Then write the reflection in the tab *Reflect and export* and export the learning log. The weekly project of Week 5, described on the [week page](README.md), builds an automatic STEMI alarm on folds 9 and 10 of PTB-XL and judges it as a chest pain unit would.

<div align="center">

---

**Prof. Dr. Utku Kose**<br>
Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University<br>
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists, Turkish Society of Cardiology. First version: November 2026. Content CC BY 4.0, code MIT.*

</div>